This notebook contains all analyses connecting IL-10-induced STAT signaling with transcriptional responses, using the Cui et al. scRNA-seq cytokine perturbation dataset (Immune Dictionary) and independent bulk RNA-seq datasets for validation. The notebook contains analyses on:

- Correlation of simulated pSTAT levels with transcriptional changes in the pre-established Immune Dictionary IL-10 DEG set (Figure 4A, Appendix Figure S15)
- Identification of IL-10-responsive genes using gene-specific pSTAT–dRNA correlations (Figures 4B,C)
- Functional comparison of the Immune Dictionary and pSTAT-correlation DEG sets by over-representation analysis (Figure 4D, Appendix Figures S17–S19)
- Comparison of the magnitude of the transcriptional response between the two DEG sets to explain their limited overlap (Appendix Figure S20)
- pSTAT–dRNA relationships for individual representative genes (Appendix Figure S21)
- Estimation of gene-specific pSTAT activation thresholds by fitting sigmoid response curves to genes from both IL-10 DEG sets (Figure 4E)
- Clustering of genes according to their pSTAT activation thresholds and characterization of early-, intermediate-, and high-pSTAT transcriptional programs (Appendix Figures S22,S23)
- Validation of pSTAT-dependent transcriptional programs using WT and IL-10 variants with imapired or enhanced-signaling in CD8+ T-cell bulk RNA-seq data from Saxton et al. (Figure 4F and Figure EV4)
- Validation of the identified transcriptional programs in monocyte bulk RNA-seq data from Gorby et al. using WT IL-10 and R5A11D (Appendix Figure S24)

In [38]:
from scipy.stats import spearmanr,pearsonr,mannwhitneyu
from scipy.optimize import curve_fit
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
from collections import Counter
import numpy as np
import os
import pickle as pkl
from pydeseq2.preprocessing import deseq2_norm
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats
from pydeseq2.utils import load_example_data
import gseapy as gp
import matplotlib.colors as mcolor
from sklearn.metrics import mean_squared_error,r2_score
import sys
from sklearn.decomposition import PCA
from matplotlib.lines import Line2D
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import single, fcluster
from scipy.spatial.distance import pdist
from statsmodels.nonparametric.smoothers_lowess import lowess
from scipy import stats
import warnings

In [2]:
# Get simulation data and sum pSTAT
df_sim3 = pd.read_csv("results/immune_dict/simulations_cytk_dict_RAp_ODE.csv")
df_sim3 = df_sim3.loc[df_sim3["IL0"]==2.437003460544915e-06]
df_sim1 = pd.read_csv("results/immune_dict/simulations_cytk_dict_RAp_ODE_S1.csv")
df_sim1 = df_sim1.loc[df_sim1["IL0"]==2.437003460544915e-06]
df_sim3["Result"] = df_sim3["Result"]+df_sim1["Result"]

## Correlating estimated pSTAT with changes in the transcriptome in preestablished DEG (Figure 4A, Appendix Figure S15)

Immune dict paper they have DEG per cell type. We take these genes and separate them into over and underexpressed by IL-10 (log2fc>0 or log2fc<0 in one cell type). We filter out cell types with less than 40 single cells in either pertrubed or control setting, normalize counts via DESeq2 and plot the pSTAT vs mean dRNA of over- and underexpressed genes separately, 

In [3]:
# Get all IL-10 DEG in Cui et. al.'s work
df_g = pd.read_excel("data/signaling/41586_2023_6816_MOESM5_ESM.xlsx", sheet_name=None)
sheet_cells = [
    'B_cell',
    'cDC1',
    'cDC2',
    'eTAC',
    'ILC',
    'Macrophage',
    'MigDC',
    'Monocyte',
    'Neutrophil',
    'NK_cell',
    'pDC',
    'T_cell_CD4',
    'T_cell_CD8',
    'T_cell_gd',
    'Treg'
]
gene_up = []
gene_down = []
for cell in sheet_cells:
    gene_up = gene_up + df_g[cell].loc[(df_g[cell]["Cytokine"]=="IL-10")&(df_g[cell]["Avg_log2FC"]>0),"Gene"].to_list()
    gene_down = gene_down + df_g[cell].loc[(df_g[cell]["Cytokine"]=="IL-10")&(df_g[cell]["Avg_log2FC"]<0),"Gene"].to_list()

In [4]:
# Get scRNAseq data of mice
adata = sc.read_h5ad("data/signaling/mice_cytokine_data_full.h5ad")
# Cell type and PBS/IL10 identifier (Don't use donor information due to lack of cells)
adata.obs["Cell__Cytokine/Donor"] = adata.obs["celltype"].astype("str")+"__"+adata.obs["sample"].astype("str")
adata = adata[adata.obs["sample"].isin(["IL10","PBS"])]

# To eliminate noise, remove cell_type/sample with less than 40 perturbed single cells or control single cells
adata_IL10 = adata[adata.obs["sample"].str.contains("IL10")]
adata_PBS = adata[adata.obs["sample"].str.contains("PBS")]
df_ncellsIL10 = pd.DataFrame.from_dict(dict(Counter(adata_IL10.obs["Cell__Cytokine/Donor"])), orient='index')
df_ncellsPBS = pd.DataFrame.from_dict(dict(Counter(adata_PBS.obs["Cell__Cytokine/Donor"])), orient='index')
cells_keep = list(set([cell.split("__")[0] for cell in df_ncellsIL10.loc[df_ncellsIL10[0]>40].index]) & set([cell.split("__")[0] for cell in df_ncellsPBS.loc[df_ncellsPBS[0]>40].index]))
cell_reps_keep = df_ncellsPBS.loc[(df_ncellsPBS[0]>10)&(df_ncellsPBS.index.str.contains('|'.join(cells_keep)))].index.to_list()+df_ncellsIL10.loc[(df_ncellsIL10[0]>20)&(df_ncellsIL10.index.str.contains('|'.join(cells_keep)))].index.to_list()
adata = adata[adata.obs["Cell__Cytokine/Donor"].isin(cell_reps_keep)]

# Convert anndata into pandas dataframe
groups = adata.obs["Cell__Cytokine/Donor"]
X = adata.X
if not isinstance(X, np.ndarray):
    X = X.toarray()

# Group by donor/cytokine and sum counts (pseudobulk)
df = pd.DataFrame(X, index=adata.obs_names, columns=adata.var_names)
summed = df.groupby(groups).sum().transpose()

# Only keeping genes with more than 10 reads in total
summed = summed.loc[summed.sum(axis=1) > 0]
summed = summed.dropna(axis=0)
summed, size_factors = deseq2_norm(summed.transpose())
summed.index = [cell.split("__")[0]+"__"+cell.split("__")[1] for cell in summed.index]

# Do pseduobulking and only keep genes with more than 10 total reads
df_pseudobulk = summed.groupby(summed.index).mean().transpose()
df_pseudobulk.columns = [cell+"__Mean" for cell in df_pseudobulk.columns]
df_pseudobulk = df_pseudobulk.loc[df_pseudobulk.sum(axis=1) > 10]

# For cell type calculate difference between rest (PBS) and pertrubed state (IL10)
df_diff = pd.DataFrame((df_pseudobulk[df_pseudobulk.columns[np.arange(0,len(df_pseudobulk.columns),2)]].values-df_pseudobulk[df_pseudobulk.columns[np.arange(1,len(df_pseudobulk.columns),2)]].values), columns=df_pseudobulk.columns[np.arange(1,len(df_pseudobulk.columns),2)],index=df_pseudobulk.index)
df_sim3c = df_sim3.loc[df_sim3["Cell_type"].isin(df_diff.columns)]
df_diff = df_diff[df_sim3c["Cell_type"].values]
df_diff.loc["Result"] = df_sim3c["Result"].values

In [5]:
# Scatter plot of overespressed and underexpressed genes
fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
df_diff_up = df_diff.loc[[gene for gene in set(gene_up) if gene in df_diff.index]]
sc_up = plt.scatter(df_diff.loc["Result"], df_diff_up.clip(lower=df_diff_up.quantile(0.05), upper=df_diff_up.quantile(0.95),axis=1).mean(), color="darkred", s=100, linewidth=2)
df_diff_down = df_diff.loc[[gene for gene in set(gene_down) if gene in df_diff.index]]
sc_down = plt.scatter(df_diff.loc["Result"], df_diff_down.clip(lower=df_diff_down.quantile(0.05), upper=df_diff_down.quantile(0.95),axis=1).mean(), color="darkblue", s=100, linewidth=2)
plt.xlabel("pSTAT1+pSTAT3 Molecules", fontsize=20)
plt.ylabel(r'$\overline{\Delta \mathrm{RNA}}$', fontsize=20)
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis='x', labelsize=17)
ax.tick_params(axis='y', labelsize=17)
ax.set_xlim(-2.5, df_diff.loc["Result"].max()+10)
ymin, ymax = ax.get_ylim()
rs_up = spearmanr(df_diff.loc["Result"], df_diff_up.clip(lower=df_diff_up.quantile(0.05), upper=df_diff_up.quantile(0.95),axis=1).mean())[0]
rp_up = pearsonr(df_diff.loc["Result"], df_diff_up.clip(lower=df_diff_up.quantile(0.05), upper=df_diff_up.quantile(0.95),axis=1).mean())[0]
rs_down = spearmanr(df_diff.loc["Result"], df_diff_down.clip(lower=df_diff_down.quantile(0.05), upper=df_diff_down.quantile(0.95),axis=1).mean())[0]
rp_down = pearsonr(df_diff.loc["Result"], df_diff_down.clip(lower=df_diff_down.quantile(0.05), upper=df_diff_down.quantile(0.95),axis=1).mean())[0]
label_up = f"Overexpressed\n$r_s$={rs_up:.2f}, $r_p$={rp_up:.2f}"
label_down = f"Underexpressed\n$r_s$={rs_down:.2f}, $r_p$={rp_down:.2f}"
ax.legend([sc_up, sc_down], [label_up, label_down], fontsize=15, loc="best")
plt.savefig('figures/immune_dict/Plot_genes_immune_dict.pdf', transparent=True, bbox_inches="tight")
plt.close(fig)

In [7]:
# Scatter plot of underexpressed genes (with error bars and correlations considering all genes)
fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
df_diff_down = df_diff.loc[[gene for gene in set(gene_down) if gene in df_diff.index]]
df_diff_down = df_diff_down.clip(lower=df_diff_down.quantile(0.05), upper=df_diff_down.quantile(0.95),axis=1)
plt.errorbar(df_diff.loc["Result"], df_diff_down.mean(), yerr=df_diff_down.std(), fmt='o', color="darkblue", markersize=14, linewidth=3, capsize=5)
plt.xlabel("pSTAT1+pSTAT3 Molecules", fontsize=20)
plt.ylabel(r'$\overline{\Delta \mathrm{RNA}}$', fontsize=20)
plt.title("Underexpressed genes", fontsize=20)
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis='x', labelsize=17)
ax.tick_params(axis='y', labelsize=17)
ax.set_xlim(-2.5, df_diff.loc["Result"].max()+10)
ymin, ymax = ax.get_ylim()
rs = spearmanr(np.repeat(df_diff.loc["Result"].values,len(df_diff_down.index)),df_diff_down.transpose().values.flatten())[0]
rp = pearsonr(np.repeat(df_diff.loc["Result"].values,len(df_diff_down.index)),df_diff_down.transpose().values.flatten())[0]
plt.text(10, (ymax + ymin), f"$r_s$={rs:.2f}\n$r_p$={rp:.2f}", ha="left", va="top", fontsize=17, color="black", bbox=dict(facecolor="white", edgecolor="grey", boxstyle="round", linewidth=2, alpha=0.65), linespacing=1.1)
plt.savefig('figures/immune_dict/Plot_under_genes_immune_dict.pdf', transparent=True, bbox_inches="tight")
plt.close(fig)
# Scatter plot of overexpressed genes
fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
df_diff_up = df_diff.loc[[gene for gene in set(gene_up) if gene in df_diff.index]]
df_diff_up = df_diff_up.clip(lower=df_diff_up.quantile(0.05), upper=df_diff_up.quantile(0.95),axis=1)
plt.errorbar(df_diff.loc["Result"], df_diff_up.mean(), yerr=df_diff_up.std(), fmt='o', color="darkred", markersize=14, linewidth=3, capsize=5)
plt.scatter(df_diff.loc["Result"], df_diff_up.clip(lower=df_diff_up.quantile(0.05), upper=df_diff_up.quantile(0.95),axis=1).mean(), color="darkred", s=100, linewidth=2)
plt.xlabel("pSTAT1+pSTAT3 Molecules", fontsize=20)
plt.ylabel(r'$\overline{\Delta \mathrm{RNA}}$', fontsize=20)
plt.title("Overexpressed genes", fontsize=20)
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis='x', labelsize=17)
ax.tick_params(axis='y', labelsize=17)
ax.set_xlim(-2.5, df_diff.loc["Result"].max()+10)
ymin, ymax = ax.get_ylim()
rs = spearmanr(np.repeat(df_diff.loc["Result"].values,len(df_diff_up.index)),df_diff_up.transpose().values.flatten())[0]
rp = pearsonr(np.repeat(df_diff.loc["Result"].values,len(df_diff_up.index)),df_diff_up.transpose().values.flatten())[0]
plt.text(10, (ymax + ymin)*1.5, f"$r_s$={rs:.2f}\n$r_p$={rp:.2f}", ha="left", va="top", fontsize=17, color="black", bbox=dict(facecolor="white", edgecolor="grey", boxstyle="round", linewidth=2, alpha=0.65), linespacing=1.1)
plt.savefig('figures/immune_dict/Plot_over_genes_immune_dict.pdf', transparent=True, bbox_inches="tight")
plt.close(fig)

## DEGs from gene-specific pSTAT-dRNA correlations (Figures 4B,C)

Now that we have demostrated that model simulated pSTAT correlates with independently-derived overexpressed and underexpressed genes, can we detect other genes using gene-specific pSTAT-dRNA correlations? Filter genes by p value<0.05 and positive mean expression change for overexpressed genes and negative mean expression change for underexpressed genes

In [8]:
# Get with differential expression correlated with pSTAT
df_diff = df_diff.loc[[gene for gene in df_diff.index if gene != "Result"]]
df_diff = df_diff.clip(lower=df_diff.quantile(0.05), upper=df_diff.quantile(0.95),axis=1)
df_diff.loc["Result"] = df_sim3c["Result"].values
df_corr = pd.DataFrame(columns=["SpearmanR","pvalue"],index=df_diff.index[:-1])
for gene in df_corr.index:
    corr = spearmanr(df_diff.loc["Result"],df_diff.loc[gene])
    df_corr.loc[gene] = [corr[0],corr[1]] # No permutation test for p value since difference was minimal although the small sample size
df_corr["Amplitude"] = df_diff.max(axis=1)-df_diff.min(axis=1)
df_corr["Mean"] = df_diff.mean(axis=1)
# Permutating pSTAT values as a control
df_corr_perm = pd.DataFrame(columns=["SpearmanR","pvalue"],index=df_diff.index[:-1])
df_diff_perm = df_diff.copy()
for gene in df_corr_perm.index:
    df_diff_perm.loc[gene] = np.random.permutation(df_diff.loc[gene].values)
    corr = spearmanr(df_diff.loc["Result"],df_diff_perm.loc[gene])
    df_corr_perm.loc[gene] = [corr[0],corr[1]] # No permutation test for p value since difference was minimal although the small sample size
df_corr_perm["Amplitude"] = df_diff_perm.max(axis=1)-df_diff_perm.min(axis=1)
df_corr_perm["Mean"] = df_diff_perm.mean(axis=1)

In [12]:
# Scatter plot of underexpressed genes (mean filtered) (with error bars)
fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
df_diff_up = df_diff.loc[df_corr.loc[(df_corr["pvalue"]<0.05)&(df_corr["SpearmanR"]>0)&(df_corr["Mean"]>=0)].index.to_list()]
sc_up = plt.errorbar(df_diff.loc["Result"], df_diff_up.mean(), yerr=df_diff_up.std(), fmt='o', color="darkred", markersize=14, linewidth=3, capsize=5)
plt.xlabel("pSTAT1+pSTAT3 (Molecules)", fontsize=20)
plt.ylabel(r'$\overline{\Delta \mathrm{RNA}}$', fontsize=20)
# plt.title("Overexpressed genes", fontsize=20)
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis='x', labelsize=17)
ax.tick_params(axis='y', labelsize=17)
ax.set_xlim(-2.5, df_diff.loc["Result"].max()+10)
ymin, ymax = ax.get_ylim()
rs_up = spearmanr(np.repeat(df_diff.loc["Result"].values,len(df_diff_up.index)),df_diff_up.transpose().values.flatten())[0]
rp_up = pearsonr(np.repeat(df_diff.loc["Result"].values,len(df_diff_up.index)),df_diff_up.transpose().values.flatten())[0]
label_up = f"Overexpressed\n$r_s$={rs_up:.2f}, $r_p$={rp_up:.2f}"
ax.legend([sc_up], [label_up], fontsize=15, loc="best")
# plt.text(10, (ymax + ymin)/2, f"$r_s$={rs:.2f}\n$r_p$={rp:.2f}", ha="left", va="top", fontsize=17, color="black", bbox=dict(facecolor="white", edgecolor="grey", boxstyle="round", linewidth=2, alpha=0.65), linespacing=1.1)
plt.savefig('figures/immune_dict/Plot_over_genes_mean_filt.pdf', transparent=True, bbox_inches="tight")
plt.close(fig)
# Scatter plot of underexpressed genes (mean filtered) (with error bars)
fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
df_diff_down = df_diff.loc[df_corr.loc[(df_corr["pvalue"]<0.05)&(df_corr["SpearmanR"]<0)&(df_corr["Mean"]<=0)].index.to_list()]
sc_down = plt.errorbar(df_diff.loc["Result"], df_diff_down.mean(), yerr=df_diff_down.std(), fmt='o', color="darkblue", markersize=14, linewidth=3, capsize=5)
plt.xlabel("pSTAT1+pSTAT3 (Molecules)", fontsize=20)
plt.ylabel(r'$\overline{\Delta \mathrm{RNA}}$', fontsize=20)
# plt.title("Underexpressed genes", fontsize=20)
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis='x', labelsize=17)
ax.tick_params(axis='y', labelsize=17)
ax.set_xlim(-2.5, df_diff.loc["Result"].max()+10)
ymin, ymax = ax.get_ylim()
rs_down = spearmanr(np.repeat(df_diff.loc["Result"].values,len(df_diff_down.index)),df_diff_down.transpose().values.flatten())[0]
rp_down = pearsonr(np.repeat(df_diff.loc["Result"].values,len(df_diff_down.index)),df_diff_down.transpose().values.flatten())[0]
label_down = f"Overexpressed\n$r_s$={rs_down:.2f}, $r_p$={rp_down:.2f}"
ax.legend([sc_down], [label_down], fontsize=15, loc="best")
# plt.text(10, (ymax + ymin)/2, f"$r_s$={rs:.2f}\n$r_p$={rp:.2f}", ha="left", va="top", fontsize=17, color="black", bbox=dict(facecolor="white", edgecolor="grey", boxstyle="round", linewidth=2, alpha=0.65), linespacing=1.1)
plt.savefig('figures/immune_dict/Plot_under_genes_mean_filt.pdf', transparent=True, bbox_inches="tight")
plt.close(fig)

### Over-representation analysis (ORA) of both DEG lists (Immune Dictionary and pSTAT correlations)

Using the 2 IL-10 DEG lists (Cui et. al. and correlations), we perform an overepresentation analysis using enrichr to compare functionally both gene lists (Figure 4D and Appendix Figures S17,S18,S19)

In [21]:
# Only using genes with SpearmanR>0 and Mean>0 and SpearmanR<0 and Mean<0
bm = gp.Biomart()
m2h = bm.query(dataset='mmusculus_gene_ensembl',
               attributes=['ensembl_gene_id','external_gene_name',
                           'hsapiens_homolog_ensembl_gene',
                           'hsapiens_homolog_associated_gene_name'])
genes_human = m2h.loc[m2h["external_gene_name"].isin(df_corr.loc[(df_corr["pvalue"]<0.05)&(df_corr["SpearmanR"]<0)&(df_corr["Mean"]<=0)].index.to_list()+df_corr.loc[(df_corr["pvalue"]<0.05)&(df_corr["SpearmanR"]>0)&(df_corr["Mean"]>=0)].index.to_list()),"hsapiens_homolog_associated_gene_name"].dropna().to_list()
library_list = [
    'GO_Molecular_Function_2025',
    'GO_Biological_Process_2025'
]
# Enrichment analysis for DEG (pSTAT correlated)
sign_enrich_terms = []
for library in library_list:
    enr_corr = gp.enrichr(gene_list=genes_human,
                    gene_sets=library,
                    organism='Human',
                    background=m2h.loc[m2h["external_gene_name"].isin(adata.var.index.values),"hsapiens_homolog_associated_gene_name"].dropna().to_list(),
                    outdir=None,  # No file output
                    cutoff=1.0).results
    sign_enrich_terms.append(enr_corr)
enrich_terms = pd.concat(sign_enrich_terms)
# Also Enrichment analysis for DEG of immune dict
genes_human_id = m2h.loc[m2h["external_gene_name"].isin(list(set(gene_up+gene_down))),"hsapiens_homolog_associated_gene_name"].dropna().to_list()
sign_enrich_terms = []
for library in library_list:
    enr_corr = gp.enrichr(gene_list=genes_human_id,
                    gene_sets=library,
                    organism='Human',
                    background=m2h.loc[m2h["external_gene_name"].isin(adata.var.index.values),"hsapiens_homolog_associated_gene_name"].dropna().to_list(),
                    outdir=None,  # No file output
                    cutoff=1.0).results
    sign_enrich_terms.append(enr_corr)
enrich_terms_id = pd.concat(sign_enrich_terms)
# Also Enrichment analysis for DEG (pSTAT correlated) that are not in immune dict
genes_human_no_id = [gene for gene in genes_human if gene not in genes_human_id]
sign_enrich_terms = []
for library in library_list:
    enr_corr = gp.enrichr(gene_list=genes_human_no_id,
                    gene_sets=library,
                    organism='Human',
                    background=m2h.loc[m2h["external_gene_name"].isin(adata.var.index.values),"hsapiens_homolog_associated_gene_name"].dropna().to_list(),
                    outdir=None,  # No file output
                    cutoff=1.0).results
    sign_enrich_terms.append(enr_corr)
enrich_terms_no_id = pd.concat(sign_enrich_terms)
# Also Enrichment analysis for DEG that are only in immune dict
genes_human_only_id = [gene for gene in genes_human_id if gene not in genes_human]
sign_enrich_terms = []
for library in library_list:
    enr_corr = gp.enrichr(gene_list=genes_human_only_id,
                    gene_sets=library,
                    organism='Human',
                    background=m2h.loc[m2h["external_gene_name"].isin(adata.var.index.values),"hsapiens_homolog_associated_gene_name"].dropna().to_list(),
                    outdir=None,  # No file output
                    cutoff=1.0).results
    sign_enrich_terms.append(enr_corr)
enrich_terms_only_id = pd.concat(sign_enrich_terms)
# Also Enrichment analysis for DEG (pSTAT correlated)
genes_common = [gene for gene in genes_human if gene in genes_human_id]
sign_enrich_terms = []
for library in library_list:
    enr_corr = gp.enrichr(gene_list=genes_common,
                    gene_sets=library,
                    organism='Human',
                    background=m2h.loc[m2h["external_gene_name"].isin(adata.var.index.values),"hsapiens_homolog_associated_gene_name"].dropna().to_list(),
                    outdir=None,  # No file output
                    cutoff=1.0).results
    sign_enrich_terms.append(enr_corr)
enrich_terms_common = pd.concat(sign_enrich_terms)
# Also Enrichment analysis for DEG that are only in immune dict
genes_human_perm = m2h.loc[m2h["external_gene_name"].isin(df_corr_perm.loc[(df_corr_perm["pvalue"]<0.05)&(df_corr_perm["SpearmanR"]<0)&(df_corr_perm["Mean"]<=0)].index.to_list()+df_corr_perm.loc[(df_corr_perm["pvalue"]<0.05)&(df_corr_perm["SpearmanR"]>0)&(df_corr_perm["Mean"]>=0)].index.to_list()),"hsapiens_homolog_associated_gene_name"].dropna().to_list()
sign_enrich_terms = []
for library in library_list:
    enr_corr = gp.enrichr(gene_list=genes_human_perm,
                    gene_sets=library,
                    organism='Human',
                    background=m2h.loc[m2h["external_gene_name"].isin(adata.var.index.values),"hsapiens_homolog_associated_gene_name"].dropna().to_list(),
                    outdir=None,  # No file output
                    cutoff=1.0).results
    sign_enrich_terms.append(enr_corr)
enrich_terms_perm = pd.concat(sign_enrich_terms)

In [15]:
for library in [
    'GO_Molecular_Function_2025',
    'GO_Biological_Process_2025'
]:
    # Get enriched terms in both GSEA
    enrich_terms_plot = enrich_terms.loc[(enrich_terms["Gene_set"]==library)].sort_values(by="Combined Score")
    enrich_terms_plot_id = enrich_terms_id.loc[(enrich_terms_id["Gene_set"]==library)].sort_values(by="Combined Score")
    terms_plot = enrich_terms_plot.loc[enrich_terms_plot["Adjusted P-value"]<0.05,"Term"].tail(7).to_list()+enrich_terms_plot_id.loc[enrich_terms_plot_id["Adjusted P-value"]<0.05,"Term"].tail(7).to_list()
    enrich_terms_plot = enrich_terms_plot.loc[enrich_terms_plot["Term"].isin(terms_plot)]
    enrich_terms_plot_id = enrich_terms_plot_id.loc[enrich_terms_plot_id["Term"].isin(terms_plot)]
    # Overlap given the background in the experiment
    lib_dict = gp.get_library(name=library, organism="Human")
    query_u_bckgr = set(genes_human)&set(m2h.loc[m2h["external_gene_name"].isin(adata.var.index.values),"hsapiens_homolog_associated_gene_name"].dropna().to_list())
    enrich_terms_plot["Overlap_bckg"] = np.array([len(set(lib_dict[term])&query_u_bckgr) for term in enrich_terms_plot["Term"]])/np.array([len(set(lib_dict[term])&set(m2h.loc[m2h["external_gene_name"].isin(adata.var.index.values),"hsapiens_homolog_associated_gene_name"].dropna().to_list())) for term in enrich_terms_plot["Term"]])
    query_u_bckgr = set(genes_human_id)&set(m2h.loc[m2h["external_gene_name"].isin(adata.var.index.values),"hsapiens_homolog_associated_gene_name"].dropna().to_list())
    enrich_terms_plot_id["Overlap_bckg"] = np.array([len(set(lib_dict[term])&query_u_bckgr) for term in enrich_terms_plot_id["Term"]])/np.array([len(set(lib_dict[term])&set(m2h.loc[m2h["external_gene_name"].isin(adata.var.index.values),"hsapiens_homolog_associated_gene_name"].dropna().to_list())) for term in enrich_terms_plot_id["Term"]])
    # Get all info in one plot
    enrich_terms_plot = enrich_terms_plot[["Term","Combined Score","Overlap_bckg"]]
    enrich_terms_plot.set_index("Term",inplace=True)
    enrich_terms_plot_id = enrich_terms_plot_id[["Term","Combined Score","Overlap_bckg"]]
    enrich_terms_plot_id.set_index("Term",inplace=True)
    enrich_terms_plot_id.columns = ["Combined Score Id","Overlap_bckg Id"]
    enrich_terms_plot = enrich_terms_plot.join(enrich_terms_plot_id,how='outer').replace(np.nan,0).sort_values("Combined Score")
    
    # New figure
    y = np.arange(len(enrich_terms_plot))
    h = 0.38
    fig, ax = plt.subplots(1, 1, figsize=(6.5, 0.5*len(enrich_terms_plot)+1))
    # Two bars per term
    bars_main = ax.barh(y + h/2, enrich_terms_plot["Combined Score"],     height=h, label="Combined Score",     color="#4C78A8")
    bars_id   = ax.barh(y - h/2, enrich_terms_plot["Combined Score Id"],  height=h, label="Combined Score Id",  color="#F58518")
    # Get Overlap as text
    for i, (xa, xb, ova, ovb) in enumerate(zip(enrich_terms_plot["Combined Score"], enrich_terms_plot["Combined Score Id"],
                                               enrich_terms_plot["Overlap_bckg"], enrich_terms_plot["Overlap_bckg Id"])):
        ax.text(xa + 1, y[i] + h/2, f"{ova:.2f}", va="center", ha="left", fontsize=11)
        ax.text(xb + 1, y[i] - h/2, f"{ovb:.2f}", va="center", ha="left", fontsize=11)
    # Styling
    ax.set_yticks(y)
    ax.set_xlabel("Combined Score", fontsize=14)
    ax.spines["bottom"].set_linewidth(4)
    ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_tick_params(width=4, length=7)
    ax.yaxis.set_tick_params(width=4, length=7)
    ax.tick_params(axis='x', labelsize=14)
    ax.tick_params(axis='y', labelsize=14)
    ax.set_yticklabels(enrich_terms_plot.index, fontsize=14)
    plt.savefig('figures/immune_dict/GSEA/enrichr_results_'+library+'.pdf', transparent=True, format="pdf", bbox_inches="tight")
    plt.close()
    
    # Get enriched terms in both GSEA (excluding common genes between sets)
    enrich_terms_plot = enrich_terms_no_id.loc[(enrich_terms_no_id["Gene_set"]==library)].sort_values(by="Combined Score")
    enrich_terms_plot_id = enrich_terms_only_id.loc[(enrich_terms_only_id["Gene_set"]==library)].sort_values(by="Combined Score")
    terms_plot = enrich_terms_plot.loc[enrich_terms_plot["Adjusted P-value"]<0.05,"Term"].tail(7).to_list()+enrich_terms_plot_id.loc[enrich_terms_plot_id["Adjusted P-value"]<0.05,"Term"].tail(7).to_list()
    enrich_terms_plot = enrich_terms_plot.loc[enrich_terms_plot["Term"].isin(terms_plot)]
    enrich_terms_plot_id = enrich_terms_plot_id.loc[enrich_terms_plot_id["Term"].isin(terms_plot)]
    # Overlap given the background in the experiment
    lib_dict = gp.get_library(name=library, organism="Human")
    query_u_bckgr = set(genes_human)&set(m2h.loc[m2h["external_gene_name"].isin(adata.var.index.values),"hsapiens_homolog_associated_gene_name"].dropna().to_list())
    enrich_terms_plot["Overlap_bckg"] = np.array([len(set(lib_dict[term])&query_u_bckgr) for term in enrich_terms_plot["Term"]])/np.array([len(set(lib_dict[term])&set(m2h.loc[m2h["external_gene_name"].isin(adata.var.index.values),"hsapiens_homolog_associated_gene_name"].dropna().to_list())) for term in enrich_terms_plot["Term"]])
    query_u_bckgr = set(genes_human_id)&set(m2h.loc[m2h["external_gene_name"].isin(adata.var.index.values),"hsapiens_homolog_associated_gene_name"].dropna().to_list())
    enrich_terms_plot_id["Overlap_bckg"] = np.array([len(set(lib_dict[term])&query_u_bckgr) for term in enrich_terms_plot_id["Term"]])/np.array([len(set(lib_dict[term])&set(m2h.loc[m2h["external_gene_name"].isin(adata.var.index.values),"hsapiens_homolog_associated_gene_name"].dropna().to_list())) for term in enrich_terms_plot_id["Term"]])
    # Get all info in one plot
    enrich_terms_plot = enrich_terms_plot[["Term","Combined Score","Overlap_bckg"]]
    enrich_terms_plot.set_index("Term",inplace=True)
    enrich_terms_plot_id = enrich_terms_plot_id[["Term","Combined Score","Overlap_bckg"]]
    enrich_terms_plot_id.set_index("Term",inplace=True)
    enrich_terms_plot_id.columns = ["Combined Score Id","Overlap_bckg Id"]
    enrich_terms_plot = enrich_terms_plot.join(enrich_terms_plot_id,how='outer').replace(np.nan,0).sort_values("Combined Score")
    
    # New figure
    y = np.arange(len(enrich_terms_plot))
    h = 0.38
    fig, ax = plt.subplots(1, 1, figsize=(6.5, 0.5*len(enrich_terms_plot)+1))
    # Two bars per term
    bars_main = ax.barh(y + h/2, enrich_terms_plot["Combined Score"],     height=h, label="Combined Score",     color="#4C78A8")
    bars_id   = ax.barh(y - h/2, enrich_terms_plot["Combined Score Id"],  height=h, label="Combined Score Id",  color="#F58518")
    # Get Overlap as text
    for i, (xa, xb, ova, ovb) in enumerate(zip(enrich_terms_plot["Combined Score"], enrich_terms_plot["Combined Score Id"],
                                               enrich_terms_plot["Overlap_bckg"], enrich_terms_plot["Overlap_bckg Id"])):
        ax.text(xa + 1, y[i] + h/2, f"{ova:.2f}", va="center", ha="left", fontsize=11)
        ax.text(xb + 1, y[i] - h/2, f"{ovb:.2f}", va="center", ha="left", fontsize=11)
    # Styling
    ax.set_yticks(y)
    ax.set_xlabel("Combined Score", fontsize=14)
    ax.spines["bottom"].set_linewidth(4)
    ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_tick_params(width=4, length=7)
    ax.yaxis.set_tick_params(width=4, length=7)
    ax.tick_params(axis='x', labelsize=14)
    ax.tick_params(axis='y', labelsize=14)
    ax.set_yticklabels(enrich_terms_plot.index, fontsize=14)
    plt.savefig('figures/immune_dict/GSEA/enrichr_no_common_results_'+library+'.pdf', transparent=True, format="pdf", bbox_inches="tight")
    plt.close()
    
    # Get enriched terms in the common genes
    enrich_terms_plot = enrich_terms_common.loc[(enrich_terms_common["Gene_set"]==library)].sort_values(by="Combined Score")
    terms_plot = enrich_terms_plot.loc[enrich_terms_plot["Adjusted P-value"]<0.05,"Term"].tail(14).to_list()
    enrich_terms_plot = enrich_terms_plot.loc[enrich_terms_plot["Term"].isin(terms_plot)]
    
    # Overlap given the background in the experiment
    lib_dict = gp.get_library(name=library, organism="Human")
    query_u_bckgr = set(genes_human)&set(m2h.loc[m2h["external_gene_name"].isin(adata.var.index.values),"hsapiens_homolog_associated_gene_name"].dropna().to_list())
    enrich_terms_plot["Overlap_bckg"] = np.array([len(set(lib_dict[term])&query_u_bckgr) for term in enrich_terms_plot["Term"]])/np.array([len(set(lib_dict[term])&set(m2h.loc[m2h["external_gene_name"].isin(adata.var.index.values),"hsapiens_homolog_associated_gene_name"].dropna().to_list())) for term in enrich_terms_plot["Term"]])
    
    # Get all info in one plot
    enrich_terms_plot = enrich_terms_plot[["Term","Combined Score","Overlap_bckg"]]
    enrich_terms_plot.set_index("Term",inplace=True)
    enrich_terms_plot = enrich_terms_plot.replace(np.nan,0).sort_values("Combined Score")
    
    # New figure
    y = np.arange(len(enrich_terms_plot))
    h = 0.38
    fig, ax = plt.subplots(1, 1, figsize=(6.5, 0.5*len(enrich_terms_plot)+1))
    # Two bars per term
    bars_main = ax.barh(y + h/2, enrich_terms_plot["Combined Score"],     height=h, label="Combined Score",     color="#4C78A8")
    # Get Overlap as text
    for i, (xa, ova) in enumerate(zip(enrich_terms_plot["Combined Score"],
                                               enrich_terms_plot["Overlap_bckg"])):
        ax.text(xa + 1, y[i] + h/2, f"{ova:.2f}", va="center", ha="left", fontsize=11)
        # Styling
    ax.set_yticks(y)
    ax.set_xlabel("Combined Score", fontsize=14)
    ax.spines["bottom"].set_linewidth(4)
    ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_tick_params(width=4, length=7)
    ax.yaxis.set_tick_params(width=4, length=7)
    ax.tick_params(axis='x', labelsize=14)
    ax.tick_params(axis='y', labelsize=14)
    ax.set_yticklabels(enrich_terms_plot.index, fontsize=14)
    plt.savefig('figures/immune_dict/GSEA/enrichr_common_results_'+library+'.pdf', transparent=True, format="pdf", bbox_inches="tight")
    plt.close()

In [22]:
# Get enriched terms in pSTAT-RNA correlations vs permutated
library = 'GO_Biological_Process_2025'
enrich_terms_plot = enrich_terms.loc[(enrich_terms["Gene_set"]==library)].sort_values(by="Combined Score")
enrich_terms_plot_perm = enrich_terms_perm.loc[(enrich_terms_perm["Gene_set"]==library)].sort_values(by="Combined Score")
terms_plot = enrich_terms_plot.loc[enrich_terms_plot["Adjusted P-value"]<0.05,"Term"].tail(14).to_list()+enrich_terms_plot_perm.loc[enrich_terms_plot_perm["Adjusted P-value"]<0.05,"Term"].tail(7).to_list()
enrich_terms_plot = enrich_terms_plot.loc[enrich_terms_plot["Term"].isin(terms_plot)]
enrich_terms_plot_perm = enrich_terms_plot_perm.loc[enrich_terms_plot_perm["Term"].isin(terms_plot)]
# Overlap given the background in the experiment
lib_dict = gp.get_library(name=library, organism="Human")
query_u_bckgr = set(genes_human)&set(m2h.loc[m2h["external_gene_name"].isin(adata.var.index.values),"hsapiens_homolog_associated_gene_name"].dropna().to_list())
enrich_terms_plot["Overlap_bckg"] = np.array([len(set(lib_dict[term])&query_u_bckgr) for term in enrich_terms_plot["Term"]])/np.array([len(set(lib_dict[term])&set(m2h.loc[m2h["external_gene_name"].isin(adata.var.index.values),"hsapiens_homolog_associated_gene_name"].dropna().to_list())) for term in enrich_terms_plot["Term"]])
query_u_bckgr = set(genes_human_perm)&set(m2h.loc[m2h["external_gene_name"].isin(adata.var.index.values),"hsapiens_homolog_associated_gene_name"].dropna().to_list())
enrich_terms_plot_perm["Overlap_bckg"] = np.array([len(set(lib_dict[term])&query_u_bckgr) for term in enrich_terms_plot_perm["Term"]])/np.array([len(set(lib_dict[term])&set(m2h.loc[m2h["external_gene_name"].isin(adata.var.index.values),"hsapiens_homolog_associated_gene_name"].dropna().to_list())) for term in enrich_terms_plot_perm["Term"]])
# Get all info in one plot
enrich_terms_plot = enrich_terms_plot[["Term","Combined Score","Overlap_bckg"]]
enrich_terms_plot.set_index("Term",inplace=True)
enrich_terms_plot_perm = enrich_terms_plot_perm[["Term","Combined Score","Overlap_bckg"]]
enrich_terms_plot_perm.set_index("Term",inplace=True)
enrich_terms_plot_perm.columns = ["Combined Score perm","Overlap_bckg perm"]
enrich_terms_plot = enrich_terms_plot.join(enrich_terms_plot_perm,how='outer').replace(np.nan,0).sort_values("Combined Score")

# New figure
y = np.arange(len(enrich_terms_plot))
h = 0.38
fig, ax = plt.subplots(1, 1, figsize=(6.5, 0.5*len(enrich_terms_plot)+1))
# Two bars per term
bars_main = ax.barh(y + h/2, enrich_terms_plot["Combined Score"],     height=h, label="DEG pSTAT corr.",     color="#4C78A8")
bars_perm   = ax.barh(y - h/2, enrich_terms_plot["Combined Score perm"],  height=h, label="DEG pSTAT corr. permutated",  color="#54A24B")
# Get Overlap as text
for i, (xa, xb, ova, ovb) in enumerate(zip(enrich_terms_plot["Combined Score"], enrich_terms_plot["Combined Score perm"],
                                           enrich_terms_plot["Overlap_bckg"], enrich_terms_plot["Overlap_bckg perm"])):
    ax.text(xa + 1, y[i] + h/2, f"{ova:.2f}", va="center", ha="left", fontsize=11)
    ax.text(xb + 1, y[i] - h/2, f"{ovb:.2f}", va="center", ha="left", fontsize=11)
# Styling
ax.set_yticks(y)
ax.set_xlabel("Combined Score", fontsize=14)
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=4, length=7)
ax.yaxis.set_tick_params(width=4, length=7)
ax.tick_params(axis='x', labelsize=14)
ax.tick_params(axis='y', labelsize=14)
plt.legend(fontsize=12)
ax.set_yticklabels(enrich_terms_plot.index, fontsize=14)
plt.savefig('figures/immune_dict/GSEA/enrichr_results_'+library+'_perm.pdf', transparent=True, format="pdf", bbox_inches="tight")
plt.close()

In [23]:
# Get gene lists with mouse name
genes_mouse = df_corr.loc[(df_corr["pvalue"]<0.05)&(df_corr["SpearmanR"]<0)&(df_corr["Mean"]<=0)].index.to_list()+df_corr.loc[(df_corr["pvalue"]<0.05)&(df_corr["SpearmanR"]>0)&(df_corr["Mean"]>=0)].index.to_list()
genes_mouse_id = list(set(gene_up+gene_down))
genes_mouse_no_id = [gene for gene in genes_mouse if gene not in genes_mouse_id]
genes_mouse_only_id = [gene for gene in genes_mouse_id if gene not in genes_mouse]

print("pSTAT correlated DEG: "+str(len(genes_mouse)))
print("Cui et. al.'s DEG: "+str(len(genes_mouse_id)))
print("Genes in both groups: "+str(len(set(genes_mouse)&set(genes_mouse_id))))

pSTAT correlated DEG: 446
Cui et. al.'s DEG: 639
Genes in both groups: 81


In [24]:
# Get DEG of immune dictionary that were only detected in one cell type and print % of genes in each of the lists
genes_1cell = pd.DataFrame.from_dict(Counter(gene_up+gene_down),orient='index').loc[pd.DataFrame.from_dict(Counter(gene_up+gene_down),orient='index')[0]==1].index
genes_not1cell = pd.DataFrame.from_dict(Counter(gene_up+gene_down),orient='index').loc[pd.DataFrame.from_dict(Counter(gene_up+gene_down),orient='index')[0]!=1].index
print("% DEG Immune dict. in 1 cell in pSTAT DEG: "+str(len(set(genes_mouse)&set(genes_1cell))/len(genes_1cell)*100)+"%")
print("% DEG Immune dict. in +1 cell in pSTAT DEG: "+str(len(set(genes_mouse)&set(genes_1cell))/len(genes_not1cell)*100)+"%")

% DEG Immune dict. in 1 cell in pSTAT DEG: 11.570247933884298%
% DEG Immune dict. in +1 cell in pSTAT DEG: 36.12903225806451%


## Single gene scatter plots

pSTAT vs dRNA scatter plots for specific genes (Figure S21)

In [25]:
# RNA-pSTAT correlations of genes in immune dictionary with DE centred in one/few cell lines
df_corr_only_id = df_corr.loc[[gene for gene in genes_mouse_only_id if gene in df_corr.index]]
df_corr_only_id["Max-Median (Abs)"] = (df_diff.loc[df_corr_only_id.index].abs().max(axis=1)-df_diff.loc[df_corr_only_id.index].abs().median(axis=1))
print("Potentially DEG in one/few cell type (Epigenetics): "+str(df_corr_only_id.loc[df_corr_only_id["SpearmanR"].abs()<0.2,"Max-Median (Abs)"].sort_values(ascending=False).head(20).index.to_list()))
genes_plot_epigenetics = ['March1','Lilrb4a','Ccl2','Ccl7','Cd209a']
for gene in genes_plot_epigenetics:
    fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
    plt.scatter(df_diff.loc["Result"], df_diff.loc[gene], color="darkred", s=100, linewidth=2)
    plt.xlabel("pSTAT1+pSTAT3 (Molecules)", fontsize=20)
    plt.ylabel(r'$\Delta \mathrm{RNA}$', fontsize=20)
    plt.title(gene, fontsize=20)
    ax.spines["bottom"].set_linewidth(4)
    ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_tick_params(width=4, length=8)
    ax.yaxis.set_tick_params(width=4, length=8)
    ax.tick_params(axis='x', labelsize=17)
    ax.tick_params(axis='y', labelsize=17)
    ax.set_xlim(-2.5, df_diff.loc["Result"].max()+10)
    ymin, ymax = ax.get_ylim()
    rs = spearmanr(df_diff.loc["Result"], df_diff.loc[gene])[0]
    rp = pearsonr(df_diff.loc["Result"], df_diff.loc[gene])[0]
    plt.text(10, (ymax + ymin)/2, f"$r_s$={rs:.2f}\n$r_p$={rp:.2f}", ha="left", va="top", fontsize=17, color="black", bbox=dict(facecolor="white", edgecolor="grey", boxstyle="round", linewidth=2, alpha=0.65), linespacing=1.1)
    plt.savefig('figures/immune_dict/single_genes/Plot_1cell_'+gene+'.pdf', transparent=True, bbox_inches="tight")
    plt.close(fig)

Potentially DEG in one/few cell type (Epigenetics): ['Ifi205', 'Mcemp1', 'Hp', 'Cd180', 'Tspan4', 'Camkk2', 'Tarm1', 'Lilrb4a', 'Ccl9', 'Pltp', 'Ccl7', 'Cd33', 'Cd209a', 'Kdr', 'Cd83', 'Rnase6', 'Il6ra', 'Tmem163', 'Gbp2', 'Lilr4b']


In [26]:
# RNA-pSTAT correlations of genes in immune dictionary with suboptimal correlations
print("Potential DEG with suboptimal correlations: "+str(df_corr_only_id.loc[df_corr_only_id["SpearmanR"].abs()>=0.4].index.to_list()))
genes_plot_suboptimal_corr = ['Socs3','Etv3','Dusp1','Bcl2',"Hif1a"]
for gene in genes_plot_suboptimal_corr:
    fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
    plt.scatter(df_diff.loc["Result"], df_diff.loc[gene], color="darkred", s=100, linewidth=2)
    plt.xlabel("pSTAT1+pSTAT3 (Molecules)", fontsize=20)
    plt.ylabel(r'$\Delta \mathrm{RNA}$', fontsize=20)
    plt.title(gene, fontsize=20)
    ax.spines["bottom"].set_linewidth(4)
    ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_tick_params(width=4, length=8)
    ax.yaxis.set_tick_params(width=4, length=8)
    ax.tick_params(axis='x', labelsize=17)
    ax.tick_params(axis='y', labelsize=17)
    ax.set_xlim(-2.5, df_diff.loc["Result"].max()+10)
    ymin, ymax = ax.get_ylim()
    rs = spearmanr(df_diff.loc["Result"], df_diff.loc[gene])[0]
    rp = pearsonr(df_diff.loc["Result"], df_diff.loc[gene])[0]
    plt.text(10, (ymax + ymin)/2, f"$r_s$={rs:.2f}\n$r_p$={rp:.2f}", ha="left", va="top", fontsize=17, color="black", bbox=dict(facecolor="white", edgecolor="grey", boxstyle="round", linewidth=2, alpha=0.65), linespacing=1.1)
    plt.savefig('figures/immune_dict/single_genes/Plot_suboptimal_corr_'+gene+'.pdf', transparent=True, bbox_inches="tight")
    plt.close(fig)

Potential DEG with suboptimal correlations: ['Psmb9', 'Pdcd4', 'Rbx1', 'Rgs10', 'Hnrnpa2b1', 'Cfp', 'Ifitm2', 'Srsf6', 'Neat1', 'Cd96', 'Efhd2', 'Rhob', 'Gzma', 'Atp6v0a1', 'Dna2', 'Tm9sf2', 'Csf2rb', 'Foxp1', 'Rnf166', 'Hspa4', 'Srm', 'Ier3', 'Arhgdib', 'Trbc1', 'Dusp1', 'Ssr2', 'Gadd45g', 'Sowahc', 'Napsa', 'Ddr1', 'Pid1', 'Rab24', 'Gpr35', 'Psma3', 'Gcnt2', 'Wfdc17', 'Tmem173', 'Ikzf1', 'Polr2f', 'Tpm3', 'Btla', 'Itm2a', 'Klf4', 'Ifitm1', 'Slfn2', 'H2-DMa', 'Mrps28', 'H2afz', 'Galnt6', 'Fam129a', 'Txnrd1', 'Skint3', 'Tmem14c', 'Mbd2', 'Shc1', 'Cdk2ap2', 'Serinc5', 'Cd38', 'H2-T23', 'Ccl12', 'Cd300a', 'Cdip1', 'Hip1', 'Tspan13', 'Pou2f2', 'Pfn1', 'Sem1', 'Cdh1', 'Socs3', 'Tmem176a', 'Snrpd3', 'Spn', 'Cst7', 'Ypel3', 'G3bp1', 'Fabp5', 'Timm10b', 'Xbp1', 'Jak1', 'Sra1', 'Glipr1', 'Tapbp', 'Ccl2', 'Txk', 'Tsc22d1', 'Il4ra', 'Tpm1', 'Hspa1b', 'Pstpip1', 'Rtl8a', 'Tuba1b', 'Zdhhc23', 'Trappc5', 'Tifab', 'Crebrf', 'Pnck', 'Hspa5', 'Serpinb1a', 'Hspe1', 'Lpp', 'Srsf2', 'Syngr2', 'Selenos', 

In [27]:
# Genes not in Cui et al DEG lists to add to the supplementary
for gene in ["Ptprj","Il1rl2"]:
    fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
    plt.scatter(df_diff.loc["Result"], df_diff.loc[gene], color="darkred", s=100, linewidth=2)
    plt.xlabel("pSTAT1+pSTAT3 (Molecules)", fontsize=20)
    plt.ylabel(r'$\Delta \mathrm{RNA}$', fontsize=20)
    plt.title(gene, fontsize=20)
    ax.spines["bottom"].set_linewidth(4)
    ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_tick_params(width=4, length=8)
    ax.yaxis.set_tick_params(width=4, length=8)
    ax.tick_params(axis='x', labelsize=17)
    ax.tick_params(axis='y', labelsize=17)
    ax.set_xlim(-2.5, df_diff.loc["Result"].max()+10)
    ymin, ymax = ax.get_ylim()
    rs = spearmanr(df_diff.loc["Result"], df_diff.loc[gene])[0]
    rp = pearsonr(df_diff.loc["Result"], df_diff.loc[gene])[0]
    plt.text(10, (ymax + ymin)/2, f"$r_s$={rs:.2f}\n$r_p$={rp:.2f}", ha="left", va="top", fontsize=17, color="black", bbox=dict(facecolor="white", edgecolor="grey", boxstyle="round", linewidth=2, alpha=0.65), linespacing=1.1)
    plt.savefig('figures/immune_dict/single_genes/Plot_'+gene+'.pdf', transparent=True, bbox_inches="tight")
    plt.close(fig)

### Comparing both DEG lists via histograms

Given our methodology for discovery of DEG is more restrictive than the usual methodogies like DEseq2, how can we explain the low overlap between the 2 DEG lists? -> Appendix Figure S20

In [28]:
# Plot histogram of max dRNA for both gene sets
x1 = df_diff.loc[[g for g in genes_mouse_only_id if g in df_diff.index]].abs().max(axis=1)
x2 = df_diff.loc[genes_mouse_no_id].abs().max(axis=1)

# Compute histograms manually to mirror one of them
bins = 10
hist1, bin_edges = np.histogram(x1, bins=bins, density=True)
hist1 = hist1/np.sum(hist1)
hist2, _         = np.histogram(x2, bins=bin_edges, density=True)
hist2 = hist2/np.sum(hist2)

bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

fig, ax = plt.subplots(1, 1, figsize=(10, 6), dpi=400)

# TOP histogram (normal)
ax.bar(bin_centers, hist1,
       width=(bin_edges[1]-bin_edges[0]),
       edgecolor="black", linewidth=1.5,
       color="#F58518", label="DEG immune dictionary")

# BOTTOM histogram (mirrored)
ax.bar(bin_centers, -hist2,
       width=(bin_edges[1]-bin_edges[0]),
       edgecolor="black", linewidth=1.5,
       color="#4C78A8", label="DEG pSTAT correlated")

# Zero line
ax.axhline(0, color="black", linewidth=1.5)

# Formatting (same as your original)
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis='x', labelsize=17)
ax.tick_params(axis='y', labelsize=17)

ax.set_xlabel(r"max $\Delta \mathrm{RNA}$ across cell types", fontsize=20)
ax.set_ylabel("Density", fontsize=20)

plt.legend(loc="lower right", fontsize=15)
plt.tight_layout()
yticks = ax.get_yticks()
ax.set_yticks(np.linspace(-max(abs(yticks)),max(abs(yticks)),5))
ax.set_yticklabels([round(abs(t),3) for t in np.linspace(-max(abs(yticks)),max(abs(yticks)),5)])
plt.savefig('figures/immune_dict/Plot_hist_max_dRNA.pdf', transparent=True, bbox_inches="tight")
plt.close(fig)

## Adjust sigmoid to all genes in both IL-10 DEG lists

Given the mean gene response seems to be sigmoid-like in genes whose dRNA correlates with pSTAT, we adjust sigmnoid curves in both DEG lists to see if there are genes activated at different pSTAT ranges (Figure 4E).

In [30]:
# Define logistic model
def logistic_model(x, L, k, x0, y0):
    return y0 + L / (1 + np.exp(-k * (x - x0)))
# Fit response to logistic model usigng curve_fit, considering fit can fail
def fit_gene_response(pstat, delta_tpm):
    x = np.array(pstat)
    y = np.array(delta_tpm)
    fit_results = {'params':[np.nan,np.nan,np.nan,np.nan], 'pcov':np.ones((4, 4)) * np.nan, 'r2': 0}
    try:
        p0, k, bounds = [0, 0.05, np.mean(x),np.mean(y)], 4, [[-(max(y)-min(y)),0.005,min(x),min(y)],[max(y)-min(y),1,max(x),max(y)]]
        popt, pcov = curve_fit(logistic_model, x, y, p0=p0, bounds=bounds)
        y_pred = logistic_model(x, *popt)
        r2 = r2_score(y, y_pred)
        fit_results = {'params': popt, 'pcov':pcov, 'r2': r2}
    except Exception as e:
        pass
    try:
        mask = abs(y-y_pred)<np.median(abs(y-y_pred))*5
        x,y = x[mask],y[mask]
        p0, k, bounds = [0, 0.05, np.mean(x),np.mean(y)], 4, [[-(max(y)-min(y)),0.005,min(x),min(y)],[max(y)-min(y),1,max(x),max(y)]]
        popt, pcov = curve_fit(logistic_model, x, y, p0=p0, bounds=bounds)
        y_pred = logistic_model(x, *popt)
        r2 = r2_score(y, y_pred)
        fit_results = {'params': popt, 'pcov':pcov, 'r2': r2}
    except Exception as e:
        pass
    return fit_results

In [31]:
# Adjust to all genes in the dataset
df_model = pd.DataFrame(columns=["r2_log", "L", "k", "x0", "y0", "L_sd", "k_sd", "x0_sd", "y0_sd"])
for gene in df_diff.index:
    if gene != "Result":
        fit_results = fit_gene_response(df_diff.loc["Result"], df_diff.loc[gene])
        df_model.loc[gene] = [fit_results["r2"]]+list(fit_results['params'])+list(np.sqrt(np.diag(fit_results['pcov'])))
df_model["SpearmanR"] = df_corr["SpearmanR"]
df_model["pvalue"] = df_corr["pvalue"]
df_model["Amplitude"] = df_corr["Amplitude"]
df_model["Mean"] = df_corr["Mean"]
df_model.loc[np.isnan(df_model["r2_log"]),"r2_log"] = 0

In [32]:
genes_mouse = df_corr.loc[(df_corr["pvalue"]<0.05)&(df_corr["SpearmanR"]<0)&(df_corr["Mean"]<=0)].index.to_list()+df_corr.loc[(df_corr["pvalue"]<0.05)&(df_corr["SpearmanR"]>0)&(df_corr["Mean"]>=0)].index.to_list()
df_log = df_model.loc[df_model["r2_log"]>0.5]
genes_mouse_regr = list(df_log.loc[(df_log["L"]>0)&(df_log["Mean"]>0)].index)+list(df_log.loc[(df_log["L"]<0)&(df_log["Mean"]<0)].index)
print("Genes correlation we also detect with regression: "+str(round(len(set(genes_mouse_regr)&set(genes_mouse))/len(genes_mouse)*100,2))+"%")

Genes correlation we also detect with regression: 74.44%


In [39]:
warnings.filterwarnings("ignore")
# Get the distributions of EC50s in both DEG lists
df_log1 = df_model.loc[[g for g in set(genes_mouse_only_id) if g in df_model.index]]
df_log2 = df_model.loc[[g for g in set(genes_mouse_no_id) if g in df_model.index]]
thres = 0.33*(df_model["x0"].max()-df_model["x0"].min())
df_log1 = df_log1.loc[(df_log1["x0_sd"]<thres)&(df_log1["r2_log"]>0.5)]
df_log2 = df_log2.loc[(df_log2["x0_sd"]<thres)&(df_log2["r2_log"]>0.5)]
x1 = df_log1["x0"].dropna().values
x2 = df_log2["x0"].dropna().values

# Remove one cell type and recalcualte pSTAT50
df_log1_rem = pd.DataFrame(columns=["gene","cell_removed","r2_log","x0","x0_sd"])
for cell in df_diff.columns:
    for gene in df_log1.index:
        if gene != "Result":
            fit_results = fit_gene_response(df_diff[[col for col in df_diff.columns if col!=cell]].loc["Result"], df_diff[[col for col in df_diff.columns if col!=cell]].loc[gene])
            df_log1_rem.loc[len(df_log1_rem)] = [gene,cell, fit_results["r2"],fit_results['params'][2],np.sqrt(np.diag(fit_results['pcov']))[2]]
df_log2_rem = pd.DataFrame(columns=["gene","cell_removed","r2_log","x0","x0_sd"])
for cell in df_diff.columns:
    for gene in df_log2.index:
        if gene != "Result":
            fit_results = fit_gene_response(df_diff[[col for col in df_diff.columns if col!=cell]].loc["Result"], df_diff[[col for col in df_diff.columns if col!=cell]].loc[gene])
            df_log2_rem.loc[len(df_log2_rem)] = [gene,cell, fit_results["r2"],fit_results['params'][2],np.sqrt(np.diag(fit_results['pcov']))[2]]
            hist1_rem = np.array([list(np.histogram(df_log1_rem.loc[df_log1_rem["cell_removed"]==cell,"x0"].values, bins=bin_edges, density=True)[0]) for cell in df_diff.columns])
warnings.filterwarnings("default")

In [40]:
# Compute histograms manually to mirror one of them
bins = 20
hist1, bin_edges = np.histogram(x1, bins=bins, density=True)
hist1 = hist1/np.sum(hist1)
hist2, _         = np.histogram(x2, bins=bin_edges, density=True)
hist2 = hist2/np.sum(hist2)
hist1_rem = np.array([list(np.histogram(df_log1_rem.loc[df_log1_rem["cell_removed"]==cell,"x0"].values, bins=bin_edges, density=True)[0]/np.sum(np.histogram(df_log1_rem.loc[df_log1_rem["cell_removed"]==cell,"x0"].values, bins=bin_edges, density=True)[0])) for cell in df_diff.columns])            
hist2_rem = np.array([list(np.histogram(df_log2_rem.loc[df_log2_rem["cell_removed"]==cell,"x0"].values, bins=bin_edges, density=True)[0]/np.sum(np.histogram(df_log2_rem.loc[df_log2_rem["cell_removed"]==cell,"x0"].values, bins=bin_edges, density=True)[0])) for cell in df_diff.columns])

bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

fig, ax = plt.subplots(1, 1, figsize=(10, 6), dpi=400)

# TOP histogram (normal)
ax.bar(bin_centers, hist1,
       width=(bin_edges[1]-bin_edges[0]),
       edgecolor="black", linewidth=1.5,
       color="#F58518", label="DEG immune dictionary")

# BOTTOM histogram (mirrored)
ax.bar(bin_centers, -hist2,
       width=(bin_edges[1]-bin_edges[0]),
       edgecolor="black", linewidth=1.5,
       color="#4C78A8", label="DEG pSTAT correlated")
ax.errorbar(bin_centers, hist1, np.std(np.row_stack((hist1_rem, hist1)),axis=0), fmt='none', color='black' , ecolor='black', elinewidth=3, capsize=7, barsabove=True, capthick=2)
ax.errorbar(bin_centers, -hist2, np.std(np.row_stack((hist2_rem, hist2)),axis=0), fmt='none', color='black' , ecolor='black', elinewidth=3, capsize=7, barsabove=True, capthick=2)

# Zero line
ax.axhline(0, color="black", linewidth=1.5)

# Formatting (same as your original)
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis='x', labelsize=17)
ax.tick_params(axis='y', labelsize=17)

ax.set_xlabel("EC50", fontsize=20)
ax.set_ylabel("Density", fontsize=20)

plt.legend(loc="lower right", fontsize=15)
plt.tight_layout()
yticks = ax.get_yticks()
ax.set_yticks(np.linspace(-max(abs(yticks)),max(abs(yticks)),5))
ax.set_yticklabels([round(abs(t),3) for t in np.linspace(-max(abs(yticks)),max(abs(yticks)),5)])
plt.savefig(
    'figures/immune_dict/Plot_hist_EC50_corr_id_genes_r2_sd_filt.pdf',
    transparent=True, bbox_inches="tight"
)
plt.close(fig)

### Can we clusterize genes via correlations to see if we find the early, mid and late activation genes? (Appendix Figure S22 and S23)

In [43]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import single, fcluster
from scipy.spatial.distance import pdist, squareform
from statsmodels.nonparametric.smoothers_lowess import lowess
from scipy.cluster.hierarchy import linkage

In [45]:
# Get the dRNA from the genes where we can confidently get the pSTAT50
df_diff_up = df_diff.loc[df_corr.loc[(df_corr["pvalue"]<0.05)&(df_corr["SpearmanR"]>0)&(df_corr["Mean"]>=0)].index.to_list()]
df_diff_down = df_diff.loc[df_corr.loc[(df_corr["pvalue"]<0.05)&(df_corr["SpearmanR"]<0)&(df_corr["Mean"]<=0)].index.to_list()]
df_diff_down = -df_diff_down.copy()
df_diff_DEGs_pSTAT = pd.concat([df_diff_up,df_diff_down])
df_diff_DEGs_pSTAT = df_diff_DEGs_pSTAT.loc[[gene for gene in df_log2.index if gene in df_diff_DEGs_pSTAT.index]]
df_diff_DEGs_dict = df_diff.loc[[gene for gene in df_log1.index if gene in df_diff.index]]
df_diff_DEGs_dict.loc[list(set(df_corr.loc[df_diff_DEGs_dict.index].loc[df_corr["SpearmanR"]<0].index)&set(df_model.loc[df_diff_DEGs_dict.index].loc[df_model["L"]<0].index))] = -df_diff_DEGs_dict.loc[list(set(df_corr.loc[df_diff_DEGs_dict.index].loc[df_corr["SpearmanR"]<0].index)&set(df_model.loc[df_diff_DEGs_dict.index].loc[df_model["L"]<0].index))].copy()
df_diff_DEGs_comb = pd.concat([df_diff_DEGs_pSTAT,df_diff_DEGs_dict])

# Perform Hierarchical clustering
corr = df_diff_DEGs_comb.T.corr(method='pearson').abs()  
dist = 1 - corr
np.fill_diagonal(dist.values, 0)
condensed_dist = squareform(dist.values, checks=False) # Convert square distance matrix to condensed form
Z = linkage(condensed_dist, method="average") # Hierarchical clustering
n_clusters = 4
labels = fcluster(Z, t=n_clusters, criterion="maxclust") # Cut tree into clusters
gene_clusters = pd.Series(labels, index=corr.index, name="cluster")

In [47]:
# Cluster mean response plot
cluster_colors = {
    1: "#006400",     # cluster 1 + 2
    3: "#00bfc4",
    4: "#800080",
}

point_size = 100
point_alpha = 0.33
point_lw = 2
line_width = 6
lowess_frac = 0.8

fig, ax = plt.subplots(1, 1, figsize=(9, 8), dpi=400)

x_raw = df_diff.loc["Result"]

for cluster_id in [1, 3, 4]:

    if cluster_id == 1:
        genes = gene_clusters.loc[(gene_clusters == 1) | (gene_clusters == 2)].index
        label = "Clusters 1 & 2"
    else:
        genes = gene_clusters.loc[gene_clusters == cluster_id].index
        label = f"Cluster {cluster_id}"

    y_raw = df_diff_DEGs_comb.loc[genes].mean()

    mask = x_raw.notna() & y_raw.notna()
    x = x_raw[mask].astype(float)
    y = y_raw[mask].astype(float)

    smoothed = lowess(y, x, frac=lowess_frac, return_sorted=True)

    y_scatter = y.values - np.min(smoothed[:, 1])
    y_scatter = y_scatter / np.max(y_scatter)

    y_smooth = smoothed[:, 1] - np.min(smoothed[:, 1])
    y_smooth = y_smooth / np.max(y_smooth)

    ax.scatter(
        x, y_scatter,
        color=cluster_colors[cluster_id],
        s=point_size,
        alpha=point_alpha,
        linewidth=point_lw,
    )

    ax.plot(
        smoothed[:, 0], y_smooth,
        color=cluster_colors[cluster_id],
        linewidth=line_width,
        label=label,
    )

ax.set_xlabel("pSTAT1+pSTAT3 Molecules", fontsize=20)
ax.set_ylabel(r'$\overline{\Delta \mathrm{RNA}}$ (Normalized)', fontsize=20)

ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis="x", labelsize=17)
ax.tick_params(axis="y", labelsize=17)

ax.set_xlim(-2.5, df_diff.loc["Result"].max() + 10)
ax.set_ylim(-0.15, 1.15)

ax.legend(fontsize=15, loc="best")
plt.savefig(
    "figures/immune_dict/Plot_gene_clusters_both_lists.pdf",
    transparent=True,
    bbox_inches="tight",
)
plt.close()

In [48]:
box_colors = ["#006400", "#006400", "#00bfc4", "#800080"]  # customize

data = [
    df_model.loc[gene_clusters[gene_clusters == i].index, "x0"].dropna().values
    for i in [1, 2, 3, 4]
]

fig, ax = plt.subplots(1, 1, figsize=(6, 7), dpi=400)

bp = ax.boxplot(
    data,
    widths=0.6,
    patch_artist=True,
    boxprops=dict(edgecolor="black", linewidth=1.5),
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(color="black", linewidth=1.5),
    capprops=dict(color="black", linewidth=1.5),
)

for patch, color in zip(bp["boxes"], box_colors):
    patch.set_facecolor(color)

ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis="x", labelsize=17)
ax.tick_params(axis="y", labelsize=17)

ax.set_ylabel("pSTAT50", fontsize=20)
ax.set_xticklabels(["Cluster 1", "Cluster 2", "Cluster 3", "Cluster 4"], fontsize=17)

plt.tight_layout()
plt.savefig(
    "figures/immune_dict/Plot_box_pSTAT50_clusters.pdf",
    transparent=True,
    bbox_inches="tight",
)
plt.close()

In [58]:
warnings.filterwarnings("ignore")
# Setup for ORA
bm = gp.Biomart()
m2h = bm.query(dataset='mmusculus_gene_ensembl',
               attributes=['ensembl_gene_id','external_gene_name',
                           'hsapiens_homolog_ensembl_gene',
                           'hsapiens_homolog_associated_gene_name'])
library = 'GO_Biological_Process_2025'

# Get ORA per gene group
dict_genes = {
    "Low pSTAT50": m2h.loc[m2h["external_gene_name"].isin(df_model.loc[list(set(list(df_log1.index)+list(df_log2.index)))].loc[df_model["x0"]<120].index.to_list()),"hsapiens_homolog_associated_gene_name"].dropna().to_list(),
    "Medium pSTAT50": m2h.loc[m2h["external_gene_name"].isin(df_model.loc[list(set(list(df_log1.index)+list(df_log2.index)))].loc[(df_model["x0"]>120)&(df_model["x0"]<180)].index.to_list()),"hsapiens_homolog_associated_gene_name"].dropna().to_list(),
    "High pSTAT50":m2h.loc[m2h["external_gene_name"].isin(df_model.loc[list(set(list(df_log1.index)+list(df_log2.index)))].loc[df_model["x0"]>180].index.to_list()),"hsapiens_homolog_associated_gene_name"].dropna().to_list(),
}
dict_enrichr = {}
for genes_group in dict_genes.keys():
    dict_enrichr[genes_group] = gp.enrichr(gene_list=dict_genes[genes_group],
                    gene_sets=library,
                    organism='Human',
                    background=m2h.loc[m2h["external_gene_name"].isin(adata.var.index.values),"hsapiens_homolog_associated_gene_name"].dropna().to_list(),
                    outdir=None,  # No file output
                    cutoff=1.0).results
warnings.filterwarnings("default")

In [61]:
warnings.filterwarnings("ignore")
enrich_terms_plot = pd.DataFrame()
terms_top_3groups = list(set(dict_enrichr["Low pSTAT50"].loc[dict_enrichr["Low pSTAT50"]["Adjusted P-value"]<0.05].sort_values(by="Combined Score",ascending=False).head(5)["Term"].to_list() + dict_enrichr["Medium pSTAT50"].loc[dict_enrichr["Medium pSTAT50"]["Adjusted P-value"]<0.05].sort_values(by="Combined Score",ascending=False).head(5)["Term"].to_list() + dict_enrichr["High pSTAT50"].loc[dict_enrichr["High pSTAT50"]["Adjusted P-value"]<0.05].sort_values(by="Combined Score",ascending=False).head(5)["Term"].to_list()))
for subset in dict_enrichr.keys():
    enrich_terms = dict_enrichr[subset].loc[dict_enrichr[subset]["Term"].isin(terms_top_3groups)]
    lib_dict = gp.get_library(name=library, organism="Human")
    query_u_bckgr = set(dict_genes[subset])&set(m2h.loc[m2h["external_gene_name"].isin(adata.var.index.values),"hsapiens_homolog_associated_gene_name"].dropna().to_list())
    enrich_terms["Overlap"] = np.array([len(set(lib_dict[term])&query_u_bckgr) for term in enrich_terms["Term"]])/np.array([len(set(lib_dict[term])&set(m2h.loc[m2h["external_gene_name"].isin(adata.var.index.values),"hsapiens_homolog_associated_gene_name"].dropna().to_list())) for term in enrich_terms["Term"]])
    enrich_terms.index = enrich_terms["Term"]
    enrich_terms = enrich_terms[["Combined Score","Overlap"]]
    enrich_terms.columns = ["Combined Score__"+subset,"Overlap__"+subset]
    enrich_terms_plot = pd.concat([enrich_terms_plot,enrich_terms],axis=1, join='outer')
enrich_terms_plot = enrich_terms_plot.replace(np.nan,0)

colors = {
    "Low pSTAT50": "#006400",
    "Medium pSTAT50": "#00bfc4",
    "High pSTAT50": "#800080",
}
labels = {
    "Low pSTAT50": "Low pSTAT thres. (120<pSTAT50)",
    "Medium pSTAT50": "Medium pSTAT thres. (120<pSTAT50<180)",
    "High pSTAT50": "High pSTAT thres. (pSTAT50>180)",
}
comparisons = [
    col.replace("Combined Score__", "")
    for col in enrich_terms_plot.columns
    if col.startswith("Combined Score__")
]

y = np.arange(len(enrich_terms_plot))
n = len(comparisons)

# Make bars thicker and rows a bit more spacious
group_height = 0.90
h = group_height / n
offsets = np.linspace(group_height / 2 - h / 2, -group_height / 2 + h / 2, n)

fig, ax = plt.subplots(
    1, 1,
    figsize=(8, 0.65 * len(enrich_terms_plot) + 1)
)

score_cols = []

for comp, offset in zip(comparisons, offsets):
    score_col = f"Combined Score__{comp}"
    overlap_col = f"Overlap__{comp}"
    score_cols.append(score_col)

    if overlap_col not in enrich_terms_plot.columns:
        raise ValueError(f"Missing overlap column: {overlap_col}")

    ax.barh(
        y + offset,
        enrich_terms_plot[score_col],
        height=h,
        label=labels.get(comp, comp),
        color=colors.get(comp, None),
        edgecolor="black",     # black border around bars
        linewidth=1.5
    )

# Set x-limit before writing text so text offset scales properly
xmax = enrich_terms_plot[score_cols].max().max()
text_offset = xmax * 0.015
ax.set_xlim(0, xmax * 1.22)

# Add overlap labels
for comp, offset in zip(comparisons, offsets):
    score_col = f"Combined Score__{comp}"
    overlap_col = f"Overlap__{comp}"

    for i, (score, overlap) in enumerate(
        zip(enrich_terms_plot[score_col], enrich_terms_plot[overlap_col])
    ):
        ax.text(
            score + text_offset,
            y[i] + offset,
            f"{overlap:.2f}",
            va="center",
            ha="left",
            fontsize=10,
            bbox=dict(facecolor="none", edgecolor="none", pad=0.2)  # prevents visual overlap
        )
ax.set_yticks(y)
ax.set_yticklabels(enrich_terms_plot.index, fontsize=14)
ax.set_xlabel("Combined Score", fontsize=14)
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=4, length=7)
ax.yaxis.set_tick_params(width=4, length=7)
ax.tick_params(axis="x", labelsize=14)
ax.tick_params(axis="y", labelsize=14)
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig('figures/immune_dict/GSEA/enrichr_results_pSTAT50_comparison.pdf', transparent=True, format="pdf", bbox_inches="tight")
plt.close()
warnings.filterwarnings("default")

### CD8+ T cells pertrubation bulkRNAseq data from Saxton et. al. (Figure 4F and EV4)

From the pSTAT-dRNA, we observe saturation in the transcriptomic response at high pSTAT. Does this translate to mutants with higher receptor binding, and therefore higher signaling in certain genes? Do we see genes that are only differentially expressed in mutants with higher receptor binding due to higher pSTAT needed to activate those genes?

In [65]:
# Dataframe to convert human and mouse genes
bm = gp.Biomart()
m2h = bm.query(dataset='mmusculus_gene_ensembl',
               attributes=['ensembl_gene_id','external_gene_name',
                           'hsapiens_homolog_ensembl_gene',
                           'hsapiens_homolog_associated_gene_name'])

/home/qmarti/miniconda3/envs/env_data/lib/python3.10/site-packages/gseapy/biomart.py:282: ResourceWarning: unclosed <ssl.SSLSocket fd=78, family=AddressFamily.AF_INET, type=SocketKind.SOCK_STREAM, proto=6, laddr=('10.45.1.96', 42332), raddr=('193.62.193.83', 443)>
  df = self.query_simple(
/home/qmarti/miniconda3/envs/env_data/lib/python3.10/site-packages/gseapy/biomart.py:282: ResourceWarning: unclosed <ssl.SSLSocket fd=80, family=AddressFamily.AF_INET, type=SocketKind.SOCK_STREAM, proto=6, laddr=('10.45.1.96', 42342), raddr=('193.62.193.83', 443)>
  df = self.query_simple(


In [67]:
# Get perturbation data of CD8+ T cells
df_CD8 = pd.read_csv("data/signaling/GSE160332_070720_10_readcount_genename.txt.gz", sep="\t", compression="gzip")
df_CD8.index = df_CD8["gene_id"]
df_CD8c = df_CD8.copy()
df_CD8 = df_CD8[['Unt_1', 'BS30_1', 'BS45_1', 'BS65_1', 'Unt_2', 'BS30_2', 'BS45_2', 'BS65_2']]
df_CD8 = df_CD8.loc[df_CD8.sum(axis=1) >= 10]

In [69]:
warnings.filterwarnings("ignore")
# Calculate the DEGs of the 3 IL-10 variants (PBS as control)
df_CD8_WT = df_CD8[['Unt_1', 'BS30_1', 'Unt_2', 'BS30_2']].transpose()
df_CD8_WT.index = ["PBS_1","IL10_1","PBS_2","IL10_2"]
df_metadata = pd.DataFrame(index=["PBS_1","IL10_1","PBS_2","IL10_2"],columns=["condition"])
df_metadata["condition"] = ["PBS","IL10","PBS","IL10"]
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(
    counts=df_CD8_WT,
    metadata=df_metadata,
    design="~condition",
    refit_cooks=True,
    inference=inference
)
dds.deseq2()
ds = DeseqStats(dds, contrast=["condition", "IL10", "PBS"], inference=inference)
ds.summary()
df_DEG_WT = ds.results_df
DEG_WT = ds.results_df.loc[(ds.results_df["padj"]<0.05)&(ds.results_df["log2FoldChange"].abs()>0.25)].index.to_list()

# From Super-10 calculate the DEGs (PBS control)
df_CD8_S10 = df_CD8[['Unt_1', 'BS45_1', 'Unt_2', 'BS45_2']].transpose()
df_CD8_S10.index = ["PBS_1","IL10_1","PBS_2","IL10_2"]
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(
    counts=df_CD8_S10,
    metadata=df_metadata,
    design="~condition",
    refit_cooks=True,
    inference=inference
)
dds.deseq2()
ds = DeseqStats(dds, contrast=["condition", "IL10", "PBS"], inference=inference)
ds.summary()
df_DEG_S10 = ds.results_df
DEG_S10 = ds.results_df.loc[(ds.results_df["padj"]<0.05)&(ds.results_df["log2FoldChange"].abs()>0.25)].index.to_list()

# From 10DE calculate the DEGs (PBS control)
df_CD8_10DE = df_CD8[['Unt_1', 'BS65_1', 'Unt_2', 'BS65_2']].transpose()
df_CD8_10DE.index = ["PBS_1","IL10_1","PBS_2","IL10_2"]
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(
    counts=df_CD8_10DE,
    metadata=df_metadata,
    design="~condition",
    refit_cooks=True,
    inference=inference
)
dds.deseq2()
ds = DeseqStats(dds, contrast=["condition", "IL10", "PBS"], inference=inference)
ds.summary()
df_DEG_10DE = ds.results_df
DEG_10DE = ds.results_df.loc[(ds.results_df["padj"]<0.05)&(ds.results_df["log2FoldChange"].abs()>0.25)].index.to_list()
warnings.filterwarnings("default")

Fitting size factors...
... done in 0.01 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 4.32 seconds.

Fitting dispersion trend curve...
... done in 1.53 seconds.

Fitting MAP dispersions...
... done in 9.25 seconds.

Fitting LFCs...
... done in 4.06 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 2.52 seconds.



Log2 fold change & Wald test p-value: condition IL10 vs PBS
                   baseMean  log2FoldChange     lfcSE      stat    pvalue  \
gene_id                                                                     
ENSG00000227232   44.762688       -0.144339  0.455713 -0.316732  0.751447   
ENSG00000268903    5.065921       -0.410845  1.311528 -0.313256  0.754086   
ENSG00000269981    2.043180        1.706468  2.054741  0.830503  0.406255   
ENSG00000279457   36.713923       -0.243900  0.504871 -0.483093  0.629030   
ENSG00000228463    2.251057       -1.599985  2.069014 -0.773308  0.439340   
...                     ...             ...       ...       ...       ...   
ENSG00000278384    7.791675        0.972019  1.046320  0.928988  0.352895   
ENSG00000276345  401.705155        0.130294  0.160715  0.810718  0.417527   
ENSG00000277856    2.348053       -0.087421  2.042594 -0.042799  0.965862   
ENSG00000275063    1.521115        1.122492  2.417441  0.464331  0.642411   
ENSG00000271254 

Fitting size factors...
... done in 0.01 seconds.



...

Log2 fold change & Wald test p-value: condition IL10 vs PBS
                   baseMean  log2FoldChange     lfcSE      stat    pvalue  \
gene_id                                                                     
ENSG00000227232   45.767099        0.015317  0.443736  0.034518  0.972464   
ENSG00000268903    4.390152       -0.795485  1.384743 -0.574464  0.565654   
ENSG00000269981    0.999358        0.204163  2.757149  0.074049  0.940972   
ENSG00000279457   32.194873       -0.580308  0.550895 -1.053391  0.292162   
ENSG00000228463    2.421944       -1.025863  1.864111 -0.550323  0.582098   
...                     ...             ...       ...       ...       ...   
ENSG00000278384    8.947576        1.325745  1.000321  1.325319  0.185065   
ENSG00000276345  411.780635        0.280888  0.167133  1.680628  0.092835   
ENSG00000277856    1.159426       -3.559037  3.534863 -1.006838  0.314012   
ENSG00000275063    2.603442        2.205904  1.981126  1.113460  0.265511   
ENSG00000271254 

Fitting size factors...
... done in 0.01 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 4.76 seconds.

Fitting dispersion trend curve...
... done in 1.37 seconds.

Fitting MAP dispersions...
... done in 5.04 seconds.

Fitting LFCs...
... done in 3.60 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 2.13 seconds.



Log2 fold change & Wald test p-value: condition IL10 vs PBS
                   baseMean  log2FoldChange     lfcSE      stat    pvalue  \
gene_id                                                                     
ENSG00000227232   46.002635       -0.252675  0.445992 -0.566546  0.571023   
ENSG00000268903    3.305483       -3.648826  2.062586 -1.769054  0.076885   
ENSG00000269981    1.486571        0.935840  2.344044  0.399242  0.689715   
ENSG00000279457   31.435296       -1.047032  0.543581 -1.926176  0.054082   
ENSG00000228463    3.250923       -0.286595  1.574284 -0.182048  0.855545   
...                     ...             ...       ...       ...       ...   
ENSG00000278384    7.931856        0.869128  1.042375  0.833796  0.404396   
ENSG00000276345  410.144488        0.009717  0.154467  0.062905  0.949842   
ENSG00000277856    1.763204       -1.385250  2.213726 -0.625755  0.531476   
ENSG00000275063    1.730180        1.258797  2.304384  0.546262  0.584886   
ENSG00000271254 

In [70]:
# Get DEG in any of the 3 IL-10 variants (WT, Super-10 and 10-DE)
thres_log2fc = 0.75
signature = list(set(df_DEG_WT.loc[(df_DEG_WT["padj"]<0.05)&(df_DEG_WT["log2FoldChange"].abs()>thres_log2fc)].index)|set(df_DEG_S10.loc[(df_DEG_S10["padj"]<0.05)&(df_DEG_S10["log2FoldChange"].abs()>thres_log2fc)].index))
# Get genes that are more differenitally expressed in Super-10 than WT IL-10
# genes_DE_S10 = df_DEG_WT.loc[signature].loc[((df_DEG_WT.loc[signature,"log2FoldChange"].abs()-df_DEG_S10.loc[signature,"log2FoldChange"].abs())/df_DEG_WT.loc[signature,"log2FoldChange"].abs())<-0.33].index
genes_DE_S10 = df_DEG_WT.loc[signature].loc[(df_DEG_WT["log2FoldChange"].abs()<1.25)&(df_DEG_S10["log2FoldChange"].abs()>df_DEG_WT["log2FoldChange"].abs()*1.33)].index
# Plot log2fc of WT and Super10 over DEG in any of the 3 IL-10 variants (WT, Super-10 and 10-DE)
x_WTaS10 = df_DEG_WT.loc[[gene for gene in signature if gene not in genes_DE_S10], "log2FoldChange"]
y_WTaS10 = df_DEG_S10.loc[[gene for gene in signature if gene not in genes_DE_S10], "log2FoldChange"]
x_DE_S10 = df_DEG_WT.loc[genes_DE_S10, "log2FoldChange"]
y_DE_S10 = df_DEG_S10.loc[genes_DE_S10, "log2FoldChange"]

# Determine global min/max for symmetric square limits
all_vals = np.concatenate([x_WTaS10.values, y_WTaS10.values,x_DE_S10.values, y_DE_S10.values])
lim = np.max(np.abs(all_vals))
xmin, xmax = -lim, lim
ymin, ymax = -lim, lim
# Begin plot (WT vs Super-10)
fig, ax = plt.subplots(figsize=(9.5, 10.5), dpi=400)
ax.scatter(x_WTaS10, y_WTaS10, color="purple", s=100,label="DEG IL-10")
ax.scatter(x_DE_S10, y_DE_S10, color="#00BFC4", s=100,label="DEG Super-10 specific")
# Identity line spanning full range
ax.plot([xmin, xmax], [xmin, xmax], color="black", linewidth=4)
# Force identical limits
ax.set_xlim(xmin, xmax)
ax.set_ylim(xmin, xmax)
# Force identical scaling and square axes box
ax.set_aspect('equal', adjustable='box')
ax.set_box_aspect(1)
# Explicit identical ticks
ticks = np.linspace(-3, 3, 7)
ax.set_xticks(ticks)
ax.set_yticks(ticks)
# Styling
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=25)
ax.tick_params(axis='y', labelsize=25)
ax.set_xlabel("WT log2FC", fontsize=25)
ax.set_ylabel("Super-10 log2FC", fontsize=25)
ax.legend(loc="best", fontsize=18)
plt.savefig(
    'figures/immune_dict/Saxton_data/Plot_log2fc_WT_S10.pdf',
    transparent=True, bbox_inches="tight"
)
plt.close(fig)

In [71]:
# Get genes that as differenitally expressed in 10-DE than in WT IL-10
signature_noDE_S10 = [gene for gene in signature if gene not in genes_DE_S10]
# genes_DE_10DE = df_DEG_10DE.loc[signature_noDE_S10].loc[((df_DEG_WT.loc[signature_noDE_S10,"log2FoldChange"].abs()-df_DEG_10DE.loc[signature_noDE_S10,"log2FoldChange"].abs())/df_DEG_10DE.loc[signature_noDE_S10,"log2FoldChange"]).abs()<1.5].index
genes_DE_10DE = df_DEG_WT.loc[signature_noDE_S10].loc[(df_DEG_10DE["log2FoldChange"].abs()*2.5>df_DEG_WT["log2FoldChange"].abs())].index
# Plot log2fc of WT and 10-DE
fig, ax = plt.subplots(figsize=(9.5, 10.5), dpi=400)
ax.scatter(df_DEG_WT.loc[signature,"log2FoldChange"],df_DEG_10DE.loc[signature,"log2FoldChange"], color="purple", s=100,label="DEG IL-10")
ax.scatter(df_DEG_WT.loc[genes_DE_10DE,"log2FoldChange"],df_DEG_10DE.loc[genes_DE_10DE,"log2FoldChange"], color="darkgreen", s=100,label="DEG 10-DE")
# Identity line spanning full range
ax.plot([xmin, xmax], [xmin, xmax], color="black", linewidth=4)
# Force identical limits
ax.set_xlim(xmin, xmax)
ax.set_ylim(xmin, xmax)
# Force identical scaling and square axes box
ax.set_aspect('equal', adjustable='box')
ax.set_box_aspect(1)
# Explicit identical ticks
ticks = np.linspace(-3, 3, 7)
ax.set_xticks(ticks)
ax.set_yticks(ticks)
# Styling
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=25)
ax.tick_params(axis='y', labelsize=25)
ax.set_xlabel("WT log2FC", fontsize=25)
ax.set_ylabel("10-DE log2FC", fontsize=25)
ax.legend(loc="best", fontsize=18)
plt.savefig(
    'figures/immune_dict/Saxton_data/Plot_log2fc_WT_10DE.pdf',
    transparent=True, bbox_inches="tight"
)
plt.close(fig)

Now that we have detected genes that are more differentially expressed in Super-10 with respect to WT, we see what is their activation range.

In [72]:
x_DE_S10 = df_model.loc[[gene for gene in m2h.loc[m2h["hsapiens_homolog_ensembl_gene"].isin(genes_DE_S10),"external_gene_name"].values if gene in df_model.index],"x0"].values
x_WTaS10 = df_model.loc[[gene for gene in m2h.loc[m2h["hsapiens_homolog_ensembl_gene"].isin([gene for gene in signature if gene not in genes_DE_S10 and gene not in genes_DE_10DE]),"external_gene_name"].values if gene in df_model.index],"x0"].values

In [73]:
u_stat, p_value = stats.mannwhitneyu(x_DE_S10, x_WTaS10, alternative='greater')
print(f"Mann-Whitney U: {u_stat}")
print(f"One-sided p-value (dist1 > dist2): {p_value}")

Mann-Whitney U: 532.0
One-sided p-value (dist1 > dist2): 0.045869668009175316


In [74]:
x_DE_10DE = df_model.loc[[gene for gene in m2h.loc[m2h["hsapiens_homolog_ensembl_gene"].isin(genes_DE_10DE),"external_gene_name"].values if gene in df_model.index],"x0"].values

In [75]:
u_stat, p_value = stats.mannwhitneyu(x_WTaS10, x_DE_10DE)
print(f"Mann-Whitney U: {u_stat}")
print(f"One-sided p-value (dist1 > dist2): {p_value}")

Mann-Whitney U: 485.0
One-sided p-value (dist1 > dist2): 0.7101398060739312


In [76]:
# Get the distributions of EC50s in both DEG lists
x_DE_S10 = df_model.loc[[gene for gene in m2h.loc[m2h["hsapiens_homolog_ensembl_gene"].isin(genes_DE_S10),"external_gene_name"].values if gene in df_model.index],"x0"].values
x_WTaS10 = df_model.loc[[gene for gene in m2h.loc[m2h["hsapiens_homolog_ensembl_gene"].isin([gene for gene in signature if gene not in genes_DE_S10 and gene not in genes_DE_10DE]),"external_gene_name"].values if gene in df_model.index],"x0"].values


fig, ax = plt.subplots(1, 1, figsize=(5, 7), dpi=400)
bp = ax.boxplot(
    [x_WTaS10,x_DE_S10],
    widths=0.6,
    patch_artist=True,
    boxprops=dict(edgecolor="black", linewidth=1.5),
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(color="black", linewidth=1.5),
    capprops=dict(color="black", linewidth=1.5)
)

bp["boxes"][0].set_facecolor("purple")
bp["boxes"][1].set_facecolor("#00BFC4")

# significance bracket
y_max = max(max(x_WTaS10), max(x_DE_S10))
h = y_max * 0.05
y = y_max * 1.05

ax.plot([1, 1, 2, 2], [y, y+h, y+h, y], lw=2, c="black")
ax.text(1.5, y+h, "*", ha="center", va="bottom", fontsize=24)

# Spine formatting
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Tick formatting
ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis="x", labelsize=17)
ax.tick_params(axis="y", labelsize=17)

ax.set_ylabel("EC50 (pSTAT Abundance)", fontsize=20)
ax.set_xticklabels(["DEG IL-10", "DEG Super-10 \n exclusive"], fontsize=17)
plt.tight_layout()
plt.savefig(
    'figures/immune_dict/Saxton_data/Plot_box_EC50_WT_S10.pdf',
    transparent=True, bbox_inches="tight"
)
plt.close(fig)

In [78]:
# Get the distributions of EC50s in 10DE
x_DE_10DE = df_model.loc[[gene for gene in m2h.loc[m2h["hsapiens_homolog_ensembl_gene"].isin(genes_DE_10DE),"external_gene_name"].values if gene in df_model.index],"x0"].values

fig, ax = plt.subplots(1, 1, figsize=(5, 7), dpi=400)
bp = ax.boxplot(
    [x_WTaS10,x_DE_10DE],
    widths=0.6,
    patch_artist=True,
    boxprops=dict(edgecolor="black", linewidth=1.5),
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(color="black", linewidth=1.5),
    capprops=dict(color="black", linewidth=1.5)
)

bp["boxes"][0].set_facecolor("purple")
bp["boxes"][1].set_facecolor("darkgreen")

# Spine formatting
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Tick formatting
ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis="x", labelsize=17)
ax.tick_params(axis="y", labelsize=17)

ax.set_ylabel("EC50 (pSTAT Abundance)", fontsize=20)
ax.set_xticklabels(["DEG IL-10", "DEG 10-DE"], fontsize=17)

plt.tight_layout()
plt.savefig(
    'figures/immune_dict/Saxton_data/Plot_box_EC50_WT_10DE.pdf',
    transparent=True, bbox_inches="tight"
)
plt.close(fig)

In [79]:
# Get PCA of the dataset to see differences between PBS, WT and mutant IL-10 in a low dimensional space
df_CD8 = pd.read_csv("data/signaling/GSE160332_070720_10_readcount_genename.txt.gz", sep="\t", compression="gzip")
df_CD8.index = df_CD8["gene_id"]
df_CD8c = df_CD8.copy()
df_CD8 = df_CD8[['Unt_1', 'BS30_1', 'BS45_1', 'BS65_1', 'Unt_2', 'BS30_2', 'BS45_2', 'BS65_2']]
df_CD8 = df_CD8.loc[df_CD8.sum(axis=1) >= 20]

df_CD8, size_factors = deseq2_norm(df_CD8.transpose())

In [81]:
# Compute PCA on genes more differentially expressed in Super-10 with respect to WT IL-10
pca = PCA(n_components=3)
pca.fit(df_CD8[genes_DE_S10].values)
X_pca = pca.transform(df_CD8[genes_DE_S10].values)
print("Explained variance ratio: "+str(pca.explained_variance_ratio_))

conditions = ['PBS_1', 'WT_1', 'S10_1', '10DE_1',
        'PBS_2', 'WT_2', 'S10_2', '10DE_2']

base_colors = {
    "PBS": "darkorange",
    "WT": "purple",
    "S10": "#00BFC4",
    "10DE": "darkgreen",
}
names_legend = {
    "PBS": "PBS",
    "WT": "WT IL-10",
    "S10": "Super-10",
    "10DE": "10-DE",
}
groups = [s.split("_")[0] for s in conditions]
rep_ids = [s.split("_")[1] for s in conditions]
colors = [base_colors[g] for g in groups]

fig, ax = plt.subplots(figsize=(6, 6), dpi=300)

ax.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=colors,
    s=300,
    linewidth=0,
)

# annotate replicate number
for x, y, r in zip(X_pca[:, 0], X_pca[:, 1], rep_ids):
    ax.text(
        x, y, r,
        ha="center",
        va="center",
        fontsize=14,
        color="white",
        weight="bold",
    )

ax.set_xlabel("PC1", fontsize=20)
ax.set_ylabel("PC2", fontsize=20)

ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.set_xticks([])
ax.set_yticks([])

# simple legend (groups only)
legend_handles = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        markersize=12,
        markerfacecolor=color,
        markeredgecolor="none",
        label=names_legend[group],
    )
    for group, color in base_colors.items()
]
ax.legend(
    handles=legend_handles,
    fontsize=15,
    loc="best",
)
plt.tight_layout()
plt.savefig(
    'figures/immune_dict/Saxton_data/PCA_DEG_S10.pdf',
    transparent=True, bbox_inches="tight"
)
plt.close(fig)


Explained variance ratio: [0.97638135 0.01937577 0.0021541 ]


In [82]:
# Compute PCA on remaining genes (genes not equally differentially expressed in Super-10 with respect to WT IL-10)
pca = PCA(n_components=3)
pca.fit(df_CD8[[gene for gene in signature if gene not in genes_DE_S10]].values)
X_pca = pca.transform(df_CD8[[gene for gene in signature if gene not in genes_DE_S10]].values)
print("Explained variance ratio: "+str(pca.explained_variance_ratio_))
conditions = ['PBS_1', 'WT_1', 'S10_1', '10DE_1',
        'PBS_2', 'WT_2', 'S10_2', '10DE_2']

groups = [s.split("_")[0] for s in conditions]
rep_ids = [s.split("_")[1] for s in conditions]
colors = [base_colors[g] for g in groups]
fig, ax = plt.subplots(figsize=(6, 6), dpi=300)
ax.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=colors,
    s=300,
    linewidth=0,
)
# annotate replicate number
for x, y, r in zip(X_pca[:, 0], X_pca[:, 1], rep_ids):
    ax.text(
        x, y, r,
        ha="center",
        va="center",
        fontsize=14,
        color="white",
        weight="bold",
    )
ax.set_xlabel("PC1", fontsize=20)
ax.set_ylabel("PC2", fontsize=20)
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xticks([])
ax.set_yticks([])
# simple legend (groups only)
legend_handles = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        markersize=12,
        markerfacecolor=color,
        markeredgecolor="none",
        label=names_legend[group],
    )
    for group, color in base_colors.items()
]
ax.legend(
    handles=legend_handles,
    fontsize=15,
    loc="best",
)
plt.tight_layout()
plt.savefig(
    'figures/immune_dict/Saxton_data/PCA_DEG_WTaS10.pdf',
    transparent=True, bbox_inches="tight"
)
plt.close(fig)


Explained variance ratio: [0.99248257 0.00517036 0.00113535]


In [84]:
# Compute PCA on genes equally differentially expressed in 10-DE with respect to WT IL-10
pca = PCA(n_components=3)
pca.fit(df_CD8[genes_DE_10DE].values)
X_pca = pca.transform(df_CD8[genes_DE_10DE].values)
print("Explained variance ratio: "+str(pca.explained_variance_ratio_))

conditions = ['PBS_1', 'WT_1', 'S10_1', '10DE_1',
        'PBS_2', 'WT_2', 'S10_2', '10DE_2']

groups = [s.split("_")[0] for s in conditions]
rep_ids = [s.split("_")[1] for s in conditions]
colors = [base_colors[g] for g in groups]

fig, ax = plt.subplots(figsize=(6, 6), dpi=300)

ax.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=colors,
    s=300,
    linewidth=0,
)

# annotate replicate number
for x, y, r in zip(X_pca[:, 0], X_pca[:, 1], rep_ids):
    ax.text(
        x, y, r,
        ha="center",
        va="center",
        fontsize=14,
        color="white",
        weight="bold",
    )

ax.set_xlabel("PC1", fontsize=20)
ax.set_ylabel("PC2", fontsize=20)

ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.set_xticks([])
ax.set_yticks([])

# simple legend (groups only)
legend_handles = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        markersize=12,
        markerfacecolor=color,
        markeredgecolor="none",
        label=names_legend[group],
    )
    for group, color in base_colors.items()
]
ax.legend(
    handles=legend_handles,
    fontsize=15,
    loc="best",
)
plt.tight_layout()
plt.savefig(
    'figures/immune_dict/Saxton_data/PCA_DEG_10DE.pdf',
    transparent=True, bbox_inches="tight"
)
plt.close(fig)


Explained variance ratio: [0.94138328 0.04194062 0.00769182]


### Monocytes pertrubation bulkRNAseq data from Gorby et. al. (Appendix Figure S24)

In [85]:
# Dataframe to convert human and mouse genes
bm = gp.Biomart()
m2h = bm.query(dataset='mmusculus_gene_ensembl',
               attributes=['ensembl_gene_id','external_gene_name',
                           'hsapiens_homolog_ensembl_gene',
                           'hsapiens_homolog_associated_gene_name'])

/home/qmarti/miniconda3/envs/env_data/lib/python3.10/site-packages/gseapy/biomart.py:282: ResourceWarning: unclosed <ssl.SSLSocket fd=96, family=AddressFamily.AF_INET, type=SocketKind.SOCK_STREAM, proto=6, laddr=('10.45.1.96', 34862), raddr=('193.62.193.83', 443)>
  df = self.query_simple(
/home/qmarti/miniconda3/envs/env_data/lib/python3.10/site-packages/gseapy/biomart.py:282: ResourceWarning: unclosed <ssl.SSLSocket fd=97, family=AddressFamily.AF_INET, type=SocketKind.SOCK_STREAM, proto=6, laddr=('10.45.1.96', 34878), raddr=('193.62.193.83', 443)>
  df = self.query_simple(


In [86]:
# Get calculated log2fc from the paper (no count data available)
from pathlib import Path
folder = Path("data/signaling/Gorby_RNAseq")
# Get IL-10 WT log2fc
files_WT = [f for f in folder.iterdir() if f.is_file() and any(s in f.name for s in ["D52WTD01", "D59WTD01", "D78WTD01"])]
DEA_WT_1 = pd.read_csv(files_WT[0], sep="\t", compression="gzip")
DEA_WT_1.index = DEA_WT_1["Gene"]
DEA_WT_2 = pd.read_csv(files_WT[1], sep="\t", compression="gzip")
DEA_WT_2.index = DEA_WT_2["Gene"]
DEA_WT_3 = pd.read_csv(files_WT[2], sep="\t", compression="gzip")
DEA_WT_3.index = DEA_WT_3["Gene"]
# Get R5A11D log2fc
files_R5 = [f for f in folder.iterdir() if f.is_file() and any(s in f.name for s in ["D52R5D01", "D59R5D01", "D78R5D01"])]
DEA_R5_1 = pd.read_csv(files_R5[0], sep="\t", compression="gzip")
DEA_R5_1.index = DEA_R5_1["Gene"]
DEA_R5_2 = pd.read_csv(files_R5[1], sep="\t", compression="gzip")
DEA_R5_2.index = DEA_R5_2["Gene"]
DEA_R5_3 = pd.read_csv(files_R5[2], sep="\t", compression="gzip")
DEA_R5_3.index = DEA_R5_3["Gene"]

In [87]:
# Get common genes between dataframes (most of them are common)
common_genes = list(set(DEA_WT_1.index)&set(DEA_WT_2.index)&set(DEA_WT_3.index)&set(DEA_R5_1.index)&set(DEA_R5_2.index)&set(DEA_R5_3.index))
DEA_WT_1 = DEA_WT_1.loc[common_genes]
DEA_WT_2 = DEA_WT_2.loc[common_genes]
DEA_WT_3 = DEA_WT_3.loc[common_genes]
DEA_R5_1 = DEA_R5_1.loc[common_genes]
DEA_R5_2 = DEA_R5_2.loc[common_genes]
DEA_R5_3 = DEA_R5_3.loc[common_genes]

# Get DEG of WT IL-10 and R5A11D (higher threshold used since the perturbation is stronger than Saxton's CD8+ T cell data)
thres_log2fc = 1.25
genes_WT = DEA_WT_1.loc[(DEA_WT_1["padjust"]<0.05)&(DEA_WT_2["padjust"]<0.05)&(DEA_WT_3["padjust"]<0.05)&(((DEA_WT_1["log2foldchange"]>thres_log2fc)&(DEA_WT_2["log2foldchange"]>thres_log2fc)&(DEA_WT_3["log2foldchange"]>thres_log2fc))|((DEA_WT_1["log2foldchange"]<-thres_log2fc)&(DEA_WT_2["log2foldchange"]<-thres_log2fc)&(DEA_WT_3["log2foldchange"]<-thres_log2fc)))].index.to_list()
genes_R5 = DEA_R5_1.loc[(DEA_R5_1["padjust"]<0.05)&(DEA_R5_2["padjust"]<0.05)&(DEA_R5_3["padjust"]<0.05)&(((DEA_R5_1["log2foldchange"]>thres_log2fc)&(DEA_R5_2["log2foldchange"]>thres_log2fc)&(DEA_R5_3["log2foldchange"]>thres_log2fc))|((DEA_R5_1["log2foldchange"]<-thres_log2fc)&(DEA_R5_2["log2foldchange"]<-thres_log2fc)&(DEA_R5_3["log2foldchange"]<-thres_log2fc)))].index.to_list()
signature = list(set(genes_WT+genes_R5))

# Get median log2fc of WT IL-10 and R5A11D
df_MONO_WT = pd.DataFrame(pd.concat([DEA_WT_1.loc[signature][["log2foldchange"]],DEA_WT_2.loc[signature][["log2foldchange"]],DEA_WT_3.loc[signature][["log2foldchange"]]],axis=1).median(axis=1))
df_MONO_WT.columns = ["log2fc"]
df_MONO_WT = df_MONO_WT.loc[signature]
df_MONO_R5 = pd.DataFrame(pd.concat([DEA_R5_1.loc[signature][["log2foldchange"]],DEA_R5_2.loc[signature][["log2foldchange"]],DEA_R5_3.loc[signature][["log2foldchange"]]],axis=1).median(axis=1))
df_MONO_R5.columns = ["log2fc"]
df_MONO_R5 = df_MONO_R5.loc[signature]

In [88]:
# Plot log2fc of WT and Super10 over DEG in any of the 3 IL-10 variants (WT, Super-10 and 10-DE)
# Higher thresholds used since the perturbation is stronger than Saxton's CD8+ T cell data
genes_DE_R5 = df_MONO_WT.loc[(df_MONO_WT["log2fc"].abs()<1.25)&(df_MONO_R5["log2fc"].abs()>df_MONO_WT["log2fc"].abs()*2.66)].index

x_WTaR5 = df_MONO_WT.loc[[gene for gene in signature if gene not in genes_DE_R5], "log2fc"]
y_WTaR5 = df_MONO_R5.loc[[gene for gene in signature if gene not in genes_DE_R5], "log2fc"]
x_DE_R5 = df_MONO_WT.loc[genes_DE_R5, "log2fc"]
y_DE_R5 = df_MONO_R5.loc[genes_DE_R5, "log2fc"]

# Determine global min/max for symmetric square limits
all_vals = np.concatenate([x_WTaR5.values, y_WTaR5.values,x_DE_R5.values, y_DE_R5.values])
all_vals = all_vals[~np.isnan(all_vals)]
lim = np.max(np.abs(all_vals))
xmin, xmax = -lim, lim
ymin, ymax = -lim, lim
# Begin plot (WT vs Super-10)
fig, ax = plt.subplots(figsize=(9.5, 10.5), dpi=400)
ax.scatter(x_WTaR5, y_WTaR5, color="purple", s=100,label="DEG IL-10")
ax.scatter(x_DE_R5, y_DE_R5, color="#00BFC4", s=100,label="DEG R5A11D specific")
# Identity line spanning full range
ax.plot([xmin, xmax], [xmin, xmax], color="black", linewidth=4)
# Force identical limits
ax.set_xlim(xmin, xmax)
ax.set_ylim(xmin, xmax)
# Force identical scaling and square axes box
ax.set_aspect('equal', adjustable='box')
ax.set_box_aspect(1)
# Explicit identical ticks
ticks = np.linspace(-10, 10, 5)
ax.set_xticks(ticks)
ax.set_yticks(ticks)
# Styling
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=25)
ax.tick_params(axis='y', labelsize=25)
ax.set_xlabel("WT log2FC", fontsize=25)
ax.set_ylabel("R5A11D log2FC", fontsize=25)
ax.legend(loc="best", fontsize=18)
plt.savefig(
    'figures/immune_dict/Gorby_data/Plot_log2fc_WT_R5.pdf',
    transparent=True, bbox_inches="tight"
)
plt.close(fig)

In [89]:
# Get the distributions of EC50s in both DEG lists
x_DE_R5 = df_model.loc[[gene for gene in m2h.loc[m2h["hsapiens_homolog_ensembl_gene"].isin(genes_DE_R5),"external_gene_name"].values if gene in df_model.index],"x0"].dropna().values
x_WTaR5 = df_model.loc[[gene for gene in m2h.loc[m2h["hsapiens_homolog_ensembl_gene"].isin([gene for gene in signature if gene not in genes_DE_R5]),"external_gene_name"].values if gene in df_model.index],"x0"].dropna().values

fig, ax = plt.subplots(1, 1, figsize=(5, 7), dpi=400)
bp = ax.boxplot(
    [x_WTaR5,x_DE_R5],
    widths=0.6,
    patch_artist=True,
    boxprops=dict(edgecolor="black", linewidth=1.5),
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(color="black", linewidth=1.5),
    capprops=dict(color="black", linewidth=1.5)
)

bp["boxes"][0].set_facecolor("purple")
bp["boxes"][1].set_facecolor("#00BFC4")

# significance bracket
y_max = max(max(x_WTaR5), max(x_DE_R5))
h = y_max * 0.05
y = y_max * 1.05

ax.plot([1, 1, 2, 2], [y, y+h, y+h, y], lw=2, c="black")
ax.text(1.5, y+h, "*", ha="center", va="bottom", fontsize=24)

# Spine formatting
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Tick formatting
ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis="x", labelsize=17)
ax.tick_params(axis="y", labelsize=17)

ax.set_ylabel("EC50 (pSTAT Abundance)", fontsize=20)
ax.set_xticklabels(["DEG IL-10", "DEG R5A11D \n exclusive"], fontsize=17)
plt.tight_layout()
plt.savefig(
    'figures/immune_dict/Gorby_data/Plot_box_EC50_WT_R5.pdf',
    transparent=True, bbox_inches="tight"
)
plt.close(fig)

In [90]:
print([len(x_DE_R5),len(x_WTaR5)])
u_stat, p_value = mannwhitneyu(x_DE_R5, x_WTaR5, alternative='greater')
print(f"Mann-Whitney U: {u_stat}")
print(f"One-sided p-value (dist1 > dist2): {p_value}")

[11, 380]
Mann-Whitney U: 2696.0
One-sided p-value (dist1 > dist2): 0.050647930967416266


In [95]:
# Get expression data on all variants
df_MONO = pd.read_excel("data/signaling/GSE146438_Monocyte_Normalised_Data.xlsx")
df_MONO.index = df_MONO['Gene']
df_MONO = df_MONO[['D52WTD01.value','D59WTD01.value','D78WTD01.value','D52US.value','D59US.value','D78US.value','D52R5D01.value','D59R5D01.value','D78R5D01.value']]
df_MONO = np.log2(df_MONO/df_MONO.sum()*1e6+1)

In [96]:
# Compute PCA on genes not more differentially expressed in R5A11D with respect to WT IL-10 or viceversa
pca = PCA(n_components=3)
pca.fit(df_MONO.loc[[gene for gene in signature if gene not in genes_DE_R5]].dropna().transpose().values)
X_pca = pca.transform(df_MONO.loc[[gene for gene in signature if gene not in genes_DE_R5]].dropna().transpose().values)
print("Explained variance ratio: "+str(pca.explained_variance_ratio_))
conditions = ['WT_1', 'WT_2', 'WT_3', 'PBS_1',
        'PBS_2', 'PBS_3', 'R5A11D_1', 'R5A11D_2', 'R5A11D_3']
base_colors = {
    "PBS": "darkorange",
    "WT": "purple",
    "R5A11D": "#00BFC4"
}
groups = [s.split("_")[0] for s in conditions]
rep_ids = [s.split("_")[1] for s in conditions]
colors = [base_colors[g] for g in groups]
fig, ax = plt.subplots(figsize=(6, 6), dpi=300)
ax.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=colors,
    s=300,
    linewidth=0,
)
# annotate replicate number
for x, y, r in zip(X_pca[:, 0], X_pca[:, 1], rep_ids):
    ax.text(
        x, y, r,
        ha="center",
        va="center",
        fontsize=14,
        color="white",
        weight="bold",
    )
ax.set_xlabel("PC1", fontsize=20)
ax.set_ylabel("PC2", fontsize=20)
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xticks([])
ax.set_yticks([])
# simple legend (groups only)
legend_handles = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        markersize=12,
        markerfacecolor=color,
        markeredgecolor="none",
        label=group,
    )
    for group, color in base_colors.items()
]
ax.legend(
    handles=legend_handles,
    fontsize=15,
    loc="best",
)
plt.tight_layout()
plt.savefig(
    'figures/immune_dict/Gorby_data/PCA_DEG_Mono_WTaR5.pdf',
    transparent=True, bbox_inches="tight"
)
plt.close(fig)

Explained variance ratio: [0.73350507 0.12836941 0.07215974]


In [97]:
# Compute PCA on genes not more differentially expressed in Super-10 with respect to WT IL-10 or viceversa
pca = PCA(n_components=3)
pca.fit(df_MONO.loc[genes_DE_R5].dropna().transpose().values)
X_pca = pca.transform(df_MONO.loc[genes_DE_R5].dropna().transpose().values)
print("Explained variance ratio: "+str(pca.explained_variance_ratio_))
conditions = ['WT_1', 'WT_2', 'WT_3', 'PBS_1',
        'PBS_2', 'PBS_3', 'R5A11D_1', 'R5A11D_2', 'R5A11D_3']
base_colors = {
    "PBS": "darkorange",
    "WT": "purple",
    "R5A11D": "#00BFC4"
}
groups = [s.split("_")[0] for s in conditions]
rep_ids = [s.split("_")[1] for s in conditions]
colors = [base_colors[g] for g in groups]
fig, ax = plt.subplots(figsize=(6, 6), dpi=300)
ax.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=colors,
    s=300,
    linewidth=0,
)
# annotate replicate number
for x, y, r in zip(X_pca[:, 0], X_pca[:, 1], rep_ids):
    ax.text(
        x, y, r,
        ha="center",
        va="center",
        fontsize=14,
        color="white",
        weight="bold",
    )
ax.set_xlabel("PC1", fontsize=20)
ax.set_ylabel("PC2", fontsize=20)
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xticks([])
ax.set_yticks([])
# simple legend (groups only)
legend_handles = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        markersize=12,
        markerfacecolor=color,
        markeredgecolor="none",
        label=group,
    )
    for group, color in base_colors.items()
]
ax.legend(
    handles=legend_handles,
    fontsize=15,
    loc="best",
)
plt.tight_layout()
plt.savefig(
    'figures/immune_dict/Gorby_data/PCA_DEG_Mono_R5.pdf',
    transparent=True, bbox_inches="tight"
)
plt.close(fig)

Explained variance ratio: [0.52332414 0.24698804 0.15124797]
